In [2]:
from urllib.request import Request, urlopen

url = (
    "https://media.githubusercontent.com/media/onyekpeu/IO-VNBD/master/"
    "Synchronised%20V%20abd%20S%20datasets/Categorised%20IOVNB%20Dataset/"
    "M%20%28Driver%20B%29/S-M.csv"
)

request = Request(url, headers={"Range": "bytes=0-8191"})

with urlopen(request, timeout=30) as response:
    print("HTTP status:", response.status)
    print("Bytes served:", response.headers.get("Content-Range"))
    preview = response.read().decode("utf-8", errors="replace")

print("\n".join(preview.splitlines()[:3]))

HTTP status: 206
Bytes served: bytes 0-8191/19798721
GPS LATITUDE (degrees), GPS LONGITUDE (degrees), GPS ALTITUDE (m), GPS SPEED (Kmh), GPS ACCURACY (m), GPS ORIENTATION (°),GPS SATELLITES IN RANGE, TIME SINCE START (ms), DATE (YYYY-MO-DD HH-MI-SS_SSS), ACCELEROMETER X (m/s�) , ACCELEROMETER Y (m/s�), ACCELEROMETER Z (m/s�), GRAVITY X (m/s�), GRAVITY Y (m/s�), GRAVITY Z (m/s�), GYROSCOPE Yaw (rad/s), GYROSCOPE Pitch (rad/s), GYROSCOPE Roll (rad/s), MAGNETIC FIELD X (μT), MAGNETIC FIELD Y (μT), MAGNETIC FIELD Z (μT), ORIENTATION (Yaw) (°), ORIENTATION (Pitch) (°), ORIENTATION (Roll ) (°)
52.402565,-1.503471,144.59,5.84,4,92.89,18 / 19,4227,2019-09-07 09:13:29:506,-0.9684,-0.0227,9.9665,0.0089,-0.0009,9.8066,-0.0045,-0.1089,-0.0368,-32.06,-30.37,5,312.14,-83.12,-141.8
52.402565,-1.503471,144.59,5.84,4,92.89,18 / 19,4327,2019-09-07 09:13:29:606,-0.3069,-1.3369,9.9218,-0.0087,0.0096,9.8066,0.0214,-0.1285,-0.0056,-31.75,-30.19,5.38,311.38,-83.06,-142.46


In [3]:
import csv
from urllib.request import Request, urlopen

url = (
    "https://media.githubusercontent.com/media/onyekpeu/IO-VNBD/master/"
    "Synchronised%20V%20abd%20S%20datasets/Categorised%20IOVNB%20Dataset/"
    "M%20%28Driver%20B%29/S-M.csv"
)

size = 256 * 1024
request = Request(url, headers={"Range": f"bytes=0-{size - 1}"})

with urlopen(request, timeout=30) as response:
    raw = response.read()
    print("Status:", response.status)
    print("Range:", response.headers.get("Content-Range"))
    print("Bytes held in memory:", len(raw))

text = raw.decode("utf-8", errors="replace")
lines = text.splitlines()

# The byte range may end partway through a CSV row; discard that final fragment.
if not text.endswith(("\n", "\r")):
    lines = lines[:-1]

reader = csv.DictReader(lines)
rows = list(reader)

print("Parsed rows:", len(rows))
print("Columns:", len(reader.fieldnames or []))

times = [
    int(row["TIME SINCE START (ms)"])
    for row in rows
    if row.get("TIME SINCE START (ms)", "").strip()
]
gaps = [b - a for a, b in zip(times, times[1:])]
print("First timestamp gaps (ms):", gaps[:10])

for column in ("GPS LATITUDE (degrees)", "GPS LONGITUDE (degrees)", "GPS SPEED (Kmh)"):
    values = [row.get(column, "") for row in rows]
    changes = sum(a != b for a, b in zip(values, values[1:]))
    print(f"{column}: {changes} changes across {max(0, len(values) - 1)} adjacent rows")

Status: 206
Range: bytes 0-262143/19798721
Bytes held in memory: 262144
Parsed rows: 1409
Columns: 24
First timestamp gaps (ms): []
GPS LATITUDE (degrees): 14 changes across 1408 adjacent rows
GPS LONGITUDE (degrees): 0 changes across 1408 adjacent rows
GPS SPEED (Kmh): 0 changes across 1408 adjacent rows


In [4]:
reader = csv.DictReader(lines)
reader.fieldnames = [name.strip() for name in reader.fieldnames]
rows = list(reader)

print("Columns:", len(reader.fieldnames))
print("Timestamp column found:", "TIME SINCE START (ms)" in reader.fieldnames)

times = [
    int(row["TIME SINCE START (ms)"].strip())
    for row in rows
    if row.get("TIME SINCE START (ms)", "").strip()
]
gaps = [b - a for a, b in zip(times, times[1:])]
print("First timestamp gaps (ms):", gaps[:10])

for column in ("GPS LATITUDE (degrees)", "GPS LONGITUDE (degrees)", "GPS SPEED (Kmh)"):
    values = [row.get(column, "") for row in rows]
    changes = sum(a != b for a, b in zip(values, values[1:]))
    print(f"{column}: {changes} changes across {max(0, len(values) - 1)} adjacent rows")

Columns: 24
Timestamp column found: True
First timestamp gaps (ms): [100, 100, 100, 101, 99, 100, 100, 100, 100, 101]
GPS LATITUDE (degrees): 14 changes across 1408 adjacent rows
GPS LONGITUDE (degrees): 14 changes across 1408 adjacent rows
GPS SPEED (Kmh): 14 changes across 1408 adjacent rows


In [5]:
from urllib.request import Request, urlopen

v_url = (
    "https://media.githubusercontent.com/media/onyekpeu/IO-VNBD/master/"
    "Synchronised%20V%20abd%20S%20datasets/Categorised%20IOVNB%20Dataset/"
    "M%20%28Driver%20B%29/V-M.csv"
)

request = Request(v_url, headers={"Range": "bytes=0-8191"})

with urlopen(request, timeout=30) as response:
    raw = response.read()
    print("Status:", response.status)
    print("Range:", response.headers.get("Content-Range"))
    print("Bytes held in memory:", len(raw))

preview = raw.decode("utf-8", errors="replace")
print("\n".join(preview.splitlines()[:4]))

Status: 206
Range: bytes 0-8191/22507823
Bytes held in memory: 8192
No of GPS Satellites Available, Time Since Start of Day (seconds), Latitude (degrees), Longitude (degrees), Velocity (km/hr), Heading (degrees), Height (km), Vertical velocity (km/hr), Sample period (seconds), Steering Angle (degrees), Wheel Speed Front Left (rad/sec), Wheel Speed Front Right (rad/sec), Wheel Speed Rear Left (rad/sec), Wheel Speed Rear Right (rad/sec), Yaw Rate (deg/sec), Indicated Vehicle Speed (km/hr), Indicated Longitudinal Acceleration (g), Indicated Lateral Acceleration (g), Handbrake (0 or 1), Gear Requested (Number fof gear employed 1-5), Gear (Number fof gear employed 1-5), Engine Speed (rev/min), Coolant Temperature (degrees), Clutch Position (0 or 1), Brake Pressure (psi), Brake Position (0 or 1), Battery Voltage (volts), Air Temperature (degrees), Accelerator Pedal Position (0 or 1)
136.0,29608.4,52.4025641,-1.5034825,19.652,44.111999999999995,98.03,-0.16,0.1,240.2,20.39,20.04,20.34,19.98,-4

In [6]:
from datetime import datetime
import csv

phone_dt = datetime.strptime(
    rows[0]["DATE (YYYY-MO-DD HH-MI-SS_SSS)"],
    "%Y-%m-%d %H:%M:%S:%f"
)
phone_tod = (
    phone_dt.hour * 3600
    + phone_dt.minute * 60
    + phone_dt.second
    + phone_dt.microsecond / 1_000_000
)

v_reader = csv.DictReader(preview.splitlines())
v_reader.fieldnames = [name.strip() for name in v_reader.fieldnames]
v_rows = list(v_reader)

time_col = "Time Since Start of Day (seconds)"
v_times = []
for row in v_rows:
    try:
        v_times.append(float(row[time_col]))
    except (ValueError, TypeError, KeyError):
        pass

candidate_t = phone_tod - 3600  # provisional one-hour correction
nearest_v_t = min(v_times, key=lambda t: abs(t - candidate_t))

print("Phone time of day:", round(phone_tod, 3))
print("V-M first time of day:", round(v_times[0], 3))
print("Raw difference (s):", round(phone_tod - v_times[0], 3))
print("Phone time minus 1 hour:", round(candidate_t, 3))
print("Nearest V-M time:", round(nearest_v_t, 3))
print("Residual after that correction (s):", round(candidate_t - nearest_v_t, 3))

Phone time of day: 33209.506
V-M first time of day: 29608.4
Raw difference (s): 3601.106
Phone time minus 1 hour: 29609.506
Nearest V-M time: 29609.5
Residual after that correction (s): 0.006


In [7]:
import csv
from datetime import datetime
from urllib.request import Request, urlopen

# Uses the existing v_url and S-M `rows` from earlier cells.
size = 256 * 1024
request = Request(v_url, headers={"Range": f"bytes=0-{size - 1}"})

with urlopen(request, timeout=30) as response:
    v_raw = response.read()
    print("V-M status/range/bytes:",
          response.status, response.headers.get("Content-Range"), len(v_raw))

v_text = v_raw.decode("utf-8", errors="replace")
v_lines = v_text.splitlines()
if not v_text.endswith(("\n", "\r")):
    v_lines = v_lines[:-1]

v_reader = csv.DictReader(v_lines)
v_reader.fieldnames = [name.strip() for name in v_reader.fieldnames]
v_rows = list(v_reader)

v_time_col = "Time Since Start of Day (seconds)"
v_times = []
for row in v_rows:
    try:
        v_times.append(float(row[v_time_col]))
    except (ValueError, TypeError, KeyError):
        pass

date_col = "DATE (YYYY-MO-DD HH-MI-SS_SSS)"
s_times = []
for row in rows:
    dt = datetime.strptime(row[date_col], "%Y-%m-%d %H:%M:%S:%f")
    # Provisional one-hour correction; we are testing it, not assuming it.
    s_times.append(
        (dt.hour * 3600 + dt.minute * 60 + dt.second
         + dt.microsecond / 1_000_000) - 3600
    )

overlap = []
for s_time in s_times:
    if v_times[0] <= s_time <= v_times[-1]:
        nearest = min(v_times, key=lambda t: abs(t - s_time))
        overlap.append(abs(nearest - s_time))

if overlap:
    ordered = sorted(overlap)
    print("Overlapping S-M samples:", len(ordered))
    print("Median time mismatch (s):", round(ordered[len(ordered) // 2], 4))
    print("95th-percentile mismatch (s):",
          round(ordered[int(0.95 * (len(ordered) - 1))], 4))
    print("Maximum mismatch (s):", round(ordered[-1], 4))

V-M status/range/bytes: 206 bytes 0-262143/22507823 262144
Overlapping S-M samples: 1190
Median time mismatch (s): 0.006
95th-percentile mismatch (s): 0.007
Maximum mismatch (s): 0.094


In [8]:
valid_v = []
for row in v_rows:
    try:
        valid_v.append((float(row["Time Since Start of Day (seconds)"]), row))
    except (ValueError, TypeError, KeyError):
        pass

v_ts = [time for time, _ in valid_v]
matches = []

for i, s_time in enumerate(s_times):
    if v_ts[0] <= s_time <= v_ts[-1]:
        j = min(range(len(v_ts)), key=lambda k: abs(v_ts[k] - s_time))
        matches.append((abs(v_ts[j] - s_time), i, j))

error, s_i, v_i = max(matches)

print("Largest mismatch (s):", round(error, 4))
print("S-M times around it:",
      [round(s_times[k], 3) for k in range(max(0, s_i-2), min(len(s_times), s_i+3))])
print("V-M times around it:",
      [round(v_ts[k], 3) for k in range(max(0, v_i-2), min(len(v_ts), v_i+3))])

Largest mismatch (s): 0.094
S-M times around it: [29630.306, 29630.406, 29630.506, 29630.606, 29630.707]
V-M times around it: [29630.4, 29630.4, 29630.6, 29630.7, 29630.8]


In [9]:
print("S-M times around the outlier:")
for k in range(max(0, s_i - 2), min(len(s_times), s_i + 3)):
    print(k, f"{s_times[k]:.3f}",
          rows[k].get("TIME SINCE START (ms)"))

print("\nV-M rows around the outlier:")
for k in range(max(0, v_i - 2), min(len(valid_v), v_i + 5)):
    t, row = valid_v[k]
    print(
        k, repr(t),
        "period:", row.get("Sample period (seconds)"),
        "velocity:", row.get("Velocity (km/hr)"),
        "heading:", row.get("Heading (degrees)")
    )

print("\nS-M matches over 50 ms:",
      sum(error > 0.05 for error in overlap), "of", len(overlap))

S-M times around the outlier:
208 29630.306 25027
209 29630.406 25127
210 29630.506 25227
211 29630.606 25327
212 29630.707 25428

V-M rows around the outlier:
220 29630.4 period: 0.1 velocity: 24.793000000000003 heading: 78.96300000000001
221 29630.4 period: 0.1 velocity: 24.793000000000003 heading: 78.96300000000001
222 29630.6 period: 0.2 velocity: 25.423000000000002 heading: 75.867
223 29630.7 period: 0.1 velocity: 25.736 heading: 75.42399999999999
224 29630.8 period: 0.1 velocity: 25.866 heading: 73.839
225 29630.9 period: 0.1 velocity: 26.482 heading: 72.788
226 29631.0 period: 0.1 velocity: 26.855999999999998 heading: 72.157

S-M matches over 50 ms: 1 of 1190


In [10]:
duplicate_rows = [
    row for time, row in valid_v
    if round(time, 3) == 29630.4
]

print("Rows at time 29630.4:", len(duplicate_rows))

if len(duplicate_rows) >= 2:
    first, second = duplicate_rows[:2]

    different_columns = [
        column for column in first
        if first.get(column) != second.get(column)
    ]

    print("Columns that differ:", different_columns)

    if different_columns:
        for column in different_columns:
            print(
                f"{column}: "
                f"{first.get(column)!r} vs {second.get(column)!r}"
            )
    else:
        print("These two rows are exact duplicates.")
else:
    print("Expected duplicate pair not found in the current sample.")

Rows at time 29630.4: 2
Columns that differ: []
These two rows are exact duplicates.


In [11]:
from bisect import bisect_left

# Deduplicate V-M timestamps; the repeated row was verified identical.
v_by_ms = {}
for time_s, row in valid_v:
    time_ms = round(time_s * 1000)
    v_by_ms.setdefault(time_ms, row)

v_ms = sorted(v_by_ms)

# Candidate phone-only inputs: three accelerometer and three gyroscope channels.
phone_cols = [
    column for column in rows[0]
    if isinstance(column, str)
    and column.startswith(("ACCELEROMETER ", "GYROSCOPE "))
]

aligned = []
outside_v_window = 0
too_far = 0
bad_values = 0

for s_row, s_time in zip(rows, s_times):
    s_ms = round(s_time * 1000)

    if s_ms < v_ms[0] or s_ms > v_ms[-1]:
        outside_v_window += 1
        continue

    pos = bisect_left(v_ms, s_ms)
    candidates = [k for k in (pos - 1, pos) if 0 <= k < len(v_ms)]
    best_v_ms = min((v_ms[k] for k in candidates),
                    key=lambda t: abs(t - s_ms))
    error_ms = best_v_ms - s_ms

    # Don't attach a reference reading that's more than half a 10 Hz interval away.
    if abs(error_ms) > 50:
        too_far += 1
        continue

    v_row = v_by_ms[best_v_ms]

    try:
        item = {
            "time_s": s_ms / 1000,
            "match_error_ms": error_ms,
            **{column: float(s_row[column]) for column in phone_cols},
            "ref_velocity_kmh": float(v_row["Velocity (km/hr)"]),
            "ref_heading_deg": float(v_row["Heading (degrees)"]),
        }
    except (ValueError, TypeError, KeyError):
        bad_values += 1
        continue

    aligned.append(item)

print("Phone sensor input columns:", phone_cols)
print("Duplicate V-M rows removed:", len(valid_v) - len(v_by_ms))
print("Aligned sample rows:", len(aligned))
print("Outside fetched V-M time window:", outside_v_window)
print("Rejected: timestamp mismatch over 50 ms:", too_far)
print("Rejected: missing/non-numeric values:", bad_values)
print("\nFirst three aligned rows:")
for item in aligned[:3]:
    print(item)

Phone sensor input columns: ['ACCELEROMETER X (m/s�)', 'ACCELEROMETER Y (m/s�)', 'ACCELEROMETER Z (m/s�)', 'GYROSCOPE Yaw (rad/s)', 'GYROSCOPE Pitch (rad/s)', 'GYROSCOPE Roll (rad/s)']
Duplicate V-M rows removed: 1
Aligned sample rows: 1189
Outside fetched V-M time window: 219
Rejected: timestamp mismatch over 50 ms: 1
Rejected: missing/non-numeric values: 0

First three aligned rows:
{'time_s': 29609.506, 'match_error_ms': -6, 'ACCELEROMETER X (m/s�)': -0.9684, 'ACCELEROMETER Y (m/s�)': -0.0227, 'ACCELEROMETER Z (m/s�)': 9.9665, 'GYROSCOPE Yaw (rad/s)': -0.0045, 'GYROSCOPE Pitch (rad/s)': -0.1089, 'GYROSCOPE Roll (rad/s)': -0.0368, 'ref_velocity_kmh': 21.91, 'ref_heading_deg': 52.615}
{'time_s': 29609.606, 'match_error_ms': -6, 'ACCELEROMETER X (m/s�)': -0.3069, 'ACCELEROMETER Y (m/s�)': -1.3369, 'ACCELEROMETER Z (m/s�)': 9.9218, 'GYROSCOPE Yaw (rad/s)': 0.0214, 'GYROSCOPE Pitch (rad/s)': -0.1285, 'GYROSCOPE Roll (rad/s)': -0.0056, 'ref_velocity_kmh': 22.194000000000003, 'ref_heading_

In [12]:
from collections import Counter

time_ms = [round(row["time_s"] * 1000) for row in aligned]
gaps_ms = [b - a for a, b in zip(time_ms, time_ms[1:])]

velocities = [row["ref_velocity_kmh"] for row in aligned]
headings = [row["ref_heading_deg"] for row in aligned]

print("Aligned rows:", len(aligned))
print("Time span (s):", round((time_ms[-1] - time_ms[0]) / 1000, 2))
print("Intervals between aligned rows (ms):", Counter(gaps_ms))
print("Reference velocity range (km/h):",
      round(min(velocities), 2), "to", round(max(velocities), 2))
print("Reference heading range (degrees):",
      round(min(headings), 2), "to", round(max(headings), 2))

Aligned rows: 1189
Time span (s): 118.9
Intervals between aligned rows (ms): Counter({100: 815, 101: 179, 99: 173, 98: 6, 102: 5, 104: 3, 96: 3, 97: 2, 200: 1, 103: 1})
Reference velocity range (km/h): 19.41 to 29.16
Reference heading range (degrees): 0.08 to 359.42


In [13]:
import math

headings = [row["ref_heading_deg"] for row in aligned]

# Large consecutive jumps usually indicate crossing the 0°/360° boundary.
wrap_jumps = [
    (i, headings[i - 1], headings[i], headings[i] - headings[i - 1])
    for i in range(1, len(headings))
    if abs(headings[i] - headings[i - 1]) > 180
]

print("Heading wrap jumps found:", len(wrap_jumps))
print("Examples:", wrap_jumps[:5])

# Add circular heading targets and convert speed from km/h to m/s.
for row in aligned:
    angle_rad = math.radians(row["ref_heading_deg"])
    row["target_speed_mps"] = row["ref_velocity_kmh"] / 3.6
    row["target_heading_sin"] = math.sin(angle_rad)
    row["target_heading_cos"] = math.cos(angle_rad)

print("Example targets:", {
    key: aligned[0][key]
    for key in ("target_speed_mps", "target_heading_sin", "target_heading_cos")
})

Heading wrap jumps found: 11
Examples: [(292, 359.417, 4.668, -354.74899999999997), (370, 358.341, 2.752, -355.589), (448, 356.601, 0.076, -356.525), (526, 357.959, 2.464, -355.495), (605, 357.889, 3.4010000000000002, -354.488)]
Example targets: {'target_speed_mps': 6.086111111111111, 'target_heading_sin': 0.7945736039324456, 'target_heading_cos': 0.6071678416499058}


In [14]:
# Split the aligned rows anywhere the time gap exceeds 150 ms.
segments = []
start = 0

for i in range(1, len(aligned)):
    gap_ms = round((aligned[i]["time_s"] - aligned[i - 1]["time_s"]) * 1000)
    if gap_ms > 150:
        segments.append(aligned[start:i])
        start = i

segments.append(aligned[start:])

# Build 10-reading windows and their reference targets.
window_len = 10
X_windows = []
Y_targets = []

for segment in segments:
    for end in range(window_len - 1, len(segment)):
        start_idx = end - window_len + 1

        window = [
            [float(segment[k][column]) for column in phone_cols]
            for k in range(start_idx, end + 1)
        ]

        target = segment[end]
        label = [
            target["target_speed_mps"],
            target["target_heading_sin"],
            target["target_heading_cos"],
        ]

        X_windows.append(window)
        Y_targets.append(label)

print("Continuous segment lengths:", [len(s) for s in segments])
print("Input shape:",
      (len(X_windows), window_len, len(phone_cols)))
print("Target shape:", (len(Y_targets), 3))
print("First target [speed m/s, heading sin, heading cos]:",
      Y_targets[0])


Continuous segment lengths: [210, 979]
Input shape: (1171, 10, 6)
Target shape: (1171, 3)
First target [speed m/s, heading sin, heading cos]: [6.751111111111111, 0.8223625680640901, 0.5689638008230706]


In [16]:
import numpy as np

# Flatten each 10×6 window into 60 inputs.
X = np.asarray(X_windows, dtype=np.float64).reshape(len(X_windows), -1)
Y = np.asarray(Y_targets, dtype=np.float64)

n = len(X)
purge = window_len - 1

train_end = int(0.70 * n)
val_end = int(0.85 * n)

train_idx = np.arange(0, train_end)
val_idx = np.arange(train_end + purge, val_end)
tail_idx = np.arange(val_end + purge, n)

print("Train / validation / held-out tail:",
      len(train_idx), len(val_idx), len(tail_idx))

# Standardize using training data only.
x_mean = X[train_idx].mean(axis=0)
x_std = X[train_idx].std(axis=0)
x_std[x_std < 1e-8] = 1.0
Xz = (X - x_mean) / x_std

y_mean = Y[train_idx].mean(axis=0)
y_std = Y[train_idx].std(axis=0)
y_std[y_std < 1e-8] = 1.0
Yz = (Y - y_mean) / y_std

# Fit ridge regression: the first column is the intercept.
X_train = np.column_stack([np.ones(len(train_idx)), Xz[train_idx]])
regularizer = np.eye(X_train.shape[1])
regularizer[0, 0] = 0.0  # don't penalize the intercept

alpha = 1.0
weights = np.linalg.solve(
    X_train.T @ X_train + alpha * regularizer,
    X_train.T @ Yz[train_idx]
)

def evaluate(indices, name):
    X_eval = np.column_stack([np.ones(len(indices)), Xz[indices]])
    pred = (X_eval @ weights) * y_std + y_mean
    truth = Y[indices]

    speed_mae = np.mean(np.abs(pred[:, 0] - truth[:, 0]))

    pred_deg = np.degrees(np.arctan2(pred[:, 1], pred[:, 2])) % 360
    true_deg = np.degrees(np.arctan2(truth[:, 1], truth[:, 2])) % 360
    angle_error = (pred_deg - true_deg + 180) % 360 - 180
    heading_mae = np.mean(np.abs(angle_error))

    print(
        f"{name}: speed MAE={speed_mae:.3f} m/s "
        f"({speed_mae * 3.6:.2f} km/h), "
        f"heading MAE={heading_mae:.1f}°"
    )

evaluate(val_idx, "Validation")
evaluate(tail_idx, "Held-out tail")

Train / validation / held-out tail: 819 167 167
Validation: speed MAE=0.726 m/s (2.61 km/h), heading MAE=12.7°
Held-out tail: speed MAE=0.467 m/s (1.68 km/h), heading MAE=15.6°


In [17]:
def score_predictions(y_true, y_pred):
    speed_mae = np.mean(np.abs(y_pred[:, 0] - y_true[:, 0]))

    pred_deg = np.degrees(np.arctan2(y_pred[:, 1], y_pred[:, 2])) % 360
    true_deg = np.degrees(np.arctan2(y_true[:, 1], y_true[:, 2])) % 360
    angle_error = (pred_deg - true_deg + 180) % 360 - 180

    return speed_mae, np.mean(np.abs(angle_error))

mean_target = Y[train_idx].mean(axis=0)

for name, indices in (("Validation", val_idx), ("Held-out tail", tail_idx)):
    X_eval = np.column_stack([np.ones(len(indices)), Xz[indices]])
    ridge_pred = (X_eval @ weights) * y_std + y_mean
    mean_pred = np.tile(mean_target, (len(indices), 1))
    truth = Y[indices]

    ridge_speed, ridge_heading = score_predictions(truth, ridge_pred)
    mean_speed, mean_heading = score_predictions(truth, mean_pred)

    print(
        f"{name} | ridge: {ridge_speed * 3.6:.2f} km/h, "
        f"{ridge_heading:.1f}° | training-mean baseline: "
        f"{mean_speed * 3.6:.2f} km/h, {mean_heading:.1f}°"
    )

Validation | ridge: 2.61 km/h, 12.7° | training-mean baseline: 3.19 km/h, 87.0°
Held-out tail | ridge: 1.68 km/h, 15.6° | training-mean baseline: 2.07 km/h, 99.6°


In [1]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.14.1+cpu
CUDA available: False


In [2]:
import csv
from urllib.request import Request, urlopen

S_URL = (
    "https://media.githubusercontent.com/media/onyekpeu/IO-VNBD/master/"
    "Synchronised%20V%20abd%20S%20datasets/Categorised%20IOVNB%20Dataset/"
    "M%20%28Driver%20B%29/S-M.csv"
)
V_URL = (
    "https://media.githubusercontent.com/media/onyekpeu/IO-VNBD/master/"
    "Synchronised%20V%20abd%20S%20datasets/Categorised%20IOVNB%20Dataset/"
    "M%20%28Driver%20B%29/V-M.csv"
)

N_BYTES = 1024 * 1024

def read_prefix_csv(file_url, n_bytes):
    request = Request(file_url, headers={"Range": f"bytes=0-{n_bytes - 1}"})
    with urlopen(request, timeout=60) as response:
        raw = response.read()
        print("HTTP:", response.status,
              "| range:", response.headers.get("Content-Range"),
              "| bytes:", len(raw))

    text = raw.decode("utf-8", errors="replace")
    lines = text.splitlines()

    # Drop a final row if the byte range cut through it.
    if not text.endswith(("\n", "\r")):
        lines = lines[:-1]

    reader = csv.DictReader(lines)
    reader.fieldnames = [name.strip() for name in reader.fieldnames]
    return list(reader)

s_big = read_prefix_csv(S_URL, N_BYTES)
v_big = read_prefix_csv(V_URL, N_BYTES)

s_elapsed = [int(row["TIME SINCE START (ms)"]) for row in s_big]
v_tod = [float(row["Time Since Start of Day (seconds)"]) for row in v_big]

print("S-M parsed rows:", len(s_big),
      "| elapsed-time range (ms):", s_elapsed[0], "to", s_elapsed[-1])
print("V-M parsed rows:", len(v_big),
      "| time-of-day range (s):", v_tod[0], "to", v_tod[-1])

HTTP: 206 | range: bytes 0-1048575/19798721 | bytes: 1048576
HTTP: 206 | range: bytes 0-1048575/22507823 | bytes: 1048576
S-M parsed rows: 5642 | elapsed-time range (ms): 4227 to 568327
V-M parsed rows: 4884 | time-of-day range (s): 29608.4 to 30096.7


In [3]:
import math
from bisect import bisect_left
from datetime import datetime

# Keep the same six phone channels used by the ridge baseline.
phone_cols_big = [
    name for name in s_big[0]
    if name.startswith(("ACCELEROMETER ", "GYROSCOPE "))
]

# Index V-M by milliseconds. Keep one exact duplicate; exclude any
# timestamp whose duplicate rows disagree.
v_by_ms_big = {}
identical_duplicates = 0
conflicting_times = set()

for row in v_big:
    try:
        time_ms = round(float(row["Time Since Start of Day (seconds)"]) * 1000)
    except (ValueError, TypeError, KeyError):
        continue

    if time_ms in v_by_ms_big:
        if row == v_by_ms_big[time_ms]:
            identical_duplicates += 1
        else:
            conflicting_times.add(time_ms)
    else:
        v_by_ms_big[time_ms] = row

for time_ms in conflicting_times:
    v_by_ms_big.pop(time_ms, None)

v_times_ms_big = sorted(v_by_ms_big)

# Convert S-M local phone time to the V-M time basis using the
# one-hour correction already validated on the first 8 minutes.
date_col = "DATE (YYYY-MO-DD HH-MI-SS_SSS)"
aligned_big = []
outside_v_window = 0
time_mismatch = 0
bad_rows = 0

for s_row in s_big:
    try:
        dt = datetime.strptime(s_row[date_col], "%Y-%m-%d %H:%M:%S:%f")
        s_time_ms = (
            (dt.hour * 3600 + dt.minute * 60 + dt.second) * 1000
            + dt.microsecond // 1000
            - 3_600_000
        )
    except (ValueError, TypeError, KeyError):
        bad_rows += 1
        continue

    if s_time_ms < v_times_ms_big[0] or s_time_ms > v_times_ms_big[-1]:
        outside_v_window += 1
        continue

    pos = bisect_left(v_times_ms_big, s_time_ms)
    candidates = [j for j in (pos - 1, pos) if 0 <= j < len(v_times_ms_big)]
    v_time_ms = min(candidates, key=lambda t: abs(t - s_time_ms))
    error_ms = v_time_ms - s_time_ms

    if abs(error_ms) > 50:
        time_mismatch += 1
        continue

    v_row = v_by_ms_big[v_time_ms]

    try:
        features = [float(s_row[name]) for name in phone_cols_big]
        speed_kmh = float(v_row["Velocity (km/hr)"])
        heading_deg = float(v_row["Heading (degrees)"])
        if not all(math.isfinite(x) for x in features + [speed_kmh, heading_deg]):
            bad_rows += 1
            continue
    except (ValueError, TypeError, KeyError):
        bad_rows += 1
        continue

    heading_rad = math.radians(heading_deg)
    aligned_big.append({
        "time_ms": s_time_ms,
        "time_s": s_time_ms / 1000,
        "match_error_ms": error_ms,
        **dict(zip(phone_cols_big, features)),
        "target_speed_mps": speed_kmh / 3.6,
        "target_heading_sin": math.sin(heading_rad),
        "target_heading_cos": math.cos(heading_rad),
    })

aligned_big.sort(key=lambda row: row["time_ms"])

# Split at gaps so no CNN window crosses a missing/late sample.
segments_big = []
start = 0
for i in range(1, len(aligned_big)):
    if aligned_big[i]["time_ms"] - aligned_big[i - 1]["time_ms"] > 150:
        segments_big.append(aligned_big[start:i])
        start = i
if aligned_big:
    segments_big.append(aligned_big[start:])

# Create 2-second windows: 20 readings at approximately 10 Hz.
window_len_cnn = 20
X_cnn = []
Y_cnn = []

for segment in segments_big:
    for end in range(window_len_cnn - 1, len(segment)):
        begin = end - window_len_cnn + 1
        X_cnn.append([
            [segment[k][name] for name in phone_cols_big]
            for k in range(begin, end + 1)
        ])
        target = segment[end]
        Y_cnn.append([
            target["target_speed_mps"],
            target["target_heading_sin"],
            target["target_heading_cos"],
        ])

print("Identical V-M duplicate rows:", identical_duplicates)
print("Conflicting V-M timestamps excluded:", len(conflicting_times))
print("Rows aligned:", len(aligned_big))
print("S-M rows outside this V-M prefix:", outside_v_window)
print("Rejected for time mismatch:", time_mismatch)
print("Rejected for missing/non-numeric values:", bad_rows)
print("Continuous segment lengths:", [len(s) for s in segments_big])
print("CNN inputs (N, time, channels):",
      (len(X_cnn), window_len_cnn, len(phone_cols_big)))
print("CNN targets (N, 3):", (len(Y_cnn), 3))

Identical V-M duplicate rows: 1
Conflicting V-M timestamps excluded: 0
Rows aligned: 0
S-M rows outside this V-M prefix: 770
Rejected for time mismatch: 4872
Rejected for missing/non-numeric values: 0
Continuous segment lengths: []
CNN inputs (N, time, channels): (0, 20, 6)
CNN targets (N, 3): (0, 3)


In [4]:
import math
from bisect import bisect_left
from datetime import datetime

aligned_big = []
outside_v_window = 0
time_mismatch = 0
bad_rows = 0

for s_row in s_big:
    try:
        dt = datetime.strptime(
            s_row["DATE (YYYY-MO-DD HH-MI-SS_SSS)"],
            "%Y-%m-%d %H:%M:%S:%f"
        )
        s_time_ms = (
            (dt.hour * 3600 + dt.minute * 60 + dt.second) * 1000
            + dt.microsecond // 1000
            - 3_600_000
        )
    except (ValueError, TypeError, KeyError):
        bad_rows += 1
        continue

    if s_time_ms < v_times_ms_big[0] or s_time_ms > v_times_ms_big[-1]:
        outside_v_window += 1
        continue

    pos = bisect_left(v_times_ms_big, s_time_ms)

    # Important: convert candidate indices back to timestamp values.
    candidate_times = [
        v_times_ms_big[j]
        for j in (pos - 1, pos)
        if 0 <= j < len(v_times_ms_big)
    ]
    v_time_ms = min(candidate_times, key=lambda t: abs(t - s_time_ms))
    error_ms = v_time_ms - s_time_ms

    if abs(error_ms) > 50:
        time_mismatch += 1
        continue

    v_row = v_by_ms_big[v_time_ms]

    try:
        features = [float(s_row[name]) for name in phone_cols_big]
        speed_kmh = float(v_row["Velocity (km/hr)"])
        heading_deg = float(v_row["Heading (degrees)"])

        if not all(math.isfinite(x) for x in features + [speed_kmh, heading_deg]):
            bad_rows += 1
            continue
    except (ValueError, TypeError, KeyError):
        bad_rows += 1
        continue

    heading_rad = math.radians(heading_deg)
    aligned_big.append({
        "time_ms": s_time_ms,
        "time_s": s_time_ms / 1000,
        "match_error_ms": error_ms,
        **dict(zip(phone_cols_big, features)),
        "target_speed_mps": speed_kmh / 3.6,
        "target_heading_sin": math.sin(heading_rad),
        "target_heading_cos": math.cos(heading_rad),
    })

aligned_big.sort(key=lambda row: row["time_ms"])

# Keep windows within continuous stretches; don't cross gaps over 150 ms.
segments_big = []
start = 0
for i in range(1, len(aligned_big)):
    if aligned_big[i]["time_ms"] - aligned_big[i - 1]["time_ms"] > 150:
        segments_big.append(aligned_big[start:i])
        start = i
if aligned_big:
    segments_big.append(aligned_big[start:])

# Two-second windows: 20 readings at approximately 10 Hz.
window_len_cnn = 20
X_cnn, Y_cnn = [], []

for segment in segments_big:
    for end in range(window_len_cnn - 1, len(segment)):
        begin = end - window_len_cnn + 1
        X_cnn.append([
            [segment[k][name] for name in phone_cols_big]
            for k in range(begin, end + 1)
        ])
        target = segment[end]
        Y_cnn.append([
            target["target_speed_mps"],
            target["target_heading_sin"],
            target["target_heading_cos"],
        ])

print("Rows aligned:", len(aligned_big))
print("S-M rows outside this V-M prefix:", outside_v_window)
print("Rejected for time mismatch:", time_mismatch)
print("Rejected for bad values:", bad_rows)
print("Continuous segment lengths:", [len(s) for s in segments_big])
print("CNN input shape:", (len(X_cnn), window_len_cnn, len(phone_cols_big)))
print("CNN target shape:", (len(Y_cnn), 3))

Rows aligned: 4871
S-M rows outside this V-M prefix: 770
Rejected for time mismatch: 1
Rejected for bad values: 0
Continuous segment lengths: [210, 4661]
CNN input shape: (4833, 20, 6)
CNN target shape: (4833, 3)


In [6]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# Convert to arrays. CNN input is arranged as (examples, channels, time).
X_cnn_np = np.asarray(X_cnn, dtype=np.float32)   # (N, time, channels)
Y_cnn_np = np.asarray(Y_cnn, dtype=np.float32)   # (N, 3)

n = len(X_cnn_np)
purge = window_len_cnn - 1

train_end_cnn = int(0.70 * n)
val_end_cnn = int(0.85 * n)

train_idx_cnn = np.arange(0, train_end_cnn)
val_idx_cnn = np.arange(train_end_cnn + purge, val_end_cnn)
test_idx_cnn = np.arange(val_end_cnn + purge, n)

print("Train / validation / held-out tail:",
      len(train_idx_cnn), len(val_idx_cnn), len(test_idx_cnn))

# Normalize from training data only.
x_mean_cnn = X_cnn_np[train_idx_cnn].mean(axis=(0, 1), keepdims=True)
x_std_cnn = X_cnn_np[train_idx_cnn].std(axis=(0, 1), keepdims=True)
x_std_cnn[x_std_cnn < 1e-6] = 1.0
X_norm_cnn = (X_cnn_np - x_mean_cnn) / x_std_cnn

y_mean_cnn = Y_cnn_np[train_idx_cnn].mean(axis=0, keepdims=True)
y_std_cnn = Y_cnn_np[train_idx_cnn].std(axis=0, keepdims=True)
y_std_cnn[y_std_cnn < 1e-6] = 1.0
Y_norm_cnn = (Y_cnn_np - y_mean_cnn) / y_std_cnn

X_tensor_cnn = torch.from_numpy(
    X_norm_cnn.transpose(0, 2, 1).copy()
)
Y_tensor_cnn = torch.from_numpy(Y_norm_cnn)

train_idx_tensor = torch.as_tensor(train_idx_cnn, dtype=torch.long)
val_idx_tensor = torch.as_tensor(val_idx_cnn, dtype=torch.long)
test_idx_tensor = torch.as_tensor(test_idx_cnn, dtype=torch.long)

train_loader = DataLoader(
    TensorDataset(
        X_tensor_cnn[train_idx_tensor],
        Y_tensor_cnn[train_idx_tensor],
    ),
    batch_size=128,
    shuffle=True,
)

# Small temporal CNN: six sensor channels in, three motion targets out.
class SmallIMUCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(6, 16, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(0.1),
            nn.Linear(32, 3),
        )

    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
cnn_model = SmallIMUCNN()
optimizer = torch.optim.AdamW(
    cnn_model.parameters(), lr=1e-3, weight_decay=1e-4
)
loss_fn = nn.MSELoss()

best_val_loss = float("inf")
best_state = None
best_epoch = 0
stale_epochs = 0

for epoch in range(1, 51):
    cnn_model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(cnn_model(xb), yb)
        loss.backward()
        optimizer.step()

    cnn_model.eval()
    with torch.no_grad():
        val_loss = loss_fn(
            cnn_model(X_tensor_cnn[val_idx_tensor]),
            Y_tensor_cnn[val_idx_tensor],
        ).item()

    if val_loss < best_val_loss - 1e-5:
        best_val_loss = val_loss
        best_state = {
            key: value.detach().clone()
            for key, value in cnn_model.state_dict().items()
        }
        best_epoch = epoch
        stale_epochs = 0
    else:
        stale_epochs += 1

    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch:2d} | validation loss {val_loss:.4f}")

    if stale_epochs >= 7:
        break

cnn_model.load_state_dict(best_state)
print("Best epoch:", best_epoch)

def score_cnn(y_true, y_pred):
    speed_mae = np.mean(np.abs(y_pred[:, 0] - y_true[:, 0]))

    pred_deg = np.degrees(np.arctan2(y_pred[:, 1], y_pred[:, 2])) % 360
    true_deg = np.degrees(np.arctan2(y_true[:, 1], y_true[:, 2])) % 360
    angle_error = (pred_deg - true_deg + 180) % 360 - 180

    return speed_mae, np.mean(np.abs(angle_error))

for name, idx, idx_tensor in (
    ("Validation", val_idx_cnn, val_idx_tensor),
    ("Held-out tail", test_idx_cnn, test_idx_tensor),
):
    cnn_model.eval()
    with torch.no_grad():
        pred_norm = cnn_model(X_tensor_cnn[idx_tensor]).numpy()

    pred = pred_norm * y_std_cnn + y_mean_cnn
    truth = Y_cnn_np[idx]

    speed_mae, heading_mae = score_cnn(truth, pred)
    print(
        f"{name}: speed MAE={speed_mae * 3.6:.2f} km/h, "
        f"heading MAE={heading_mae:.1f}°"
    )

Train / validation / held-out tail: 3383 706 706
Epoch  1 | validation loss 0.8043
Epoch  5 | validation loss 0.9172
Best epoch: 1
Validation: speed MAE=13.75 km/h, heading MAE=53.7°
Held-out tail: speed MAE=5.61 km/h, heading MAE=13.2°


In [7]:
def inspect_cnn_split(name, indices):
    idx_tensor = torch.as_tensor(indices, dtype=torch.long)

    cnn_model.eval()
    with torch.no_grad():
        pred_norm = cnn_model(X_tensor_cnn[idx_tensor]).numpy()

    pred = pred_norm * y_std_cnn + y_mean_cnn
    truth = Y_cnn_np[indices]

    cnn_speed_mae, cnn_heading_mae = score_cnn(truth, pred)

    mean_pred = np.tile(
        Y_cnn_np[train_idx_cnn].mean(axis=0),
        (len(indices), 1)
    )
    mean_speed_mae, mean_heading_mae = score_cnn(truth, mean_pred)

    true_speed = truth[:, 0] * 3.6
    pred_speed = pred[:, 0] * 3.6

    print(
        f"{name}: true speed range {true_speed.min():.1f}–{true_speed.max():.1f} km/h; "
        f"predicted range {pred_speed.min():.1f}–{pred_speed.max():.1f} km/h"
    )
    print(
        f"  CNN: speed MAE {cnn_speed_mae * 3.6:.2f} km/h, "
        f"heading MAE {cnn_heading_mae:.1f}°"
    )
    print(
        f"  Train-mean baseline: speed MAE {mean_speed_mae * 3.6:.2f} km/h, "
        f"heading MAE {mean_heading_mae:.1f}°"
    )

inspect_cnn_split("Train", train_idx_cnn)
inspect_cnn_split("Validation", val_idx_cnn)
inspect_cnn_split("Held-out tail", test_idx_cnn)

Train: true speed range 0.0–68.0 km/h; predicted range 28.5–46.1 km/h
  CNN: speed MAE 12.99 km/h, heading MAE 50.1°
  Train-mean baseline: speed MAE 14.61 km/h, heading MAE 74.3°
Validation: true speed range 0.0–63.8 km/h; predicted range 35.1–42.5 km/h
  CNN: speed MAE 13.75 km/h, heading MAE 53.7°
  Train-mean baseline: speed MAE 18.43 km/h, heading MAE 82.9°
Held-out tail: true speed range 26.6–46.2 km/h; predicted range 34.9–43.1 km/h
  CNN: speed MAE 5.61 km/h, heading MAE 13.2°
  Train-mean baseline: speed MAE 9.45 km/h, heading MAE 40.0°


In [8]:
# Same windows, labels, and time splits as the CNN.
X_flat_lr = X_cnn_np.reshape(len(X_cnn_np), -1)

x_mean_lr = X_flat_lr[train_idx_cnn].mean(axis=0)
x_std_lr = X_flat_lr[train_idx_cnn].std(axis=0)
x_std_lr[x_std_lr < 1e-6] = 1.0
X_norm_lr = (X_flat_lr - x_mean_lr) / x_std_lr

y_mean_lr = Y_cnn_np[train_idx_cnn].mean(axis=0, keepdims=True)
y_std_lr = Y_cnn_np[train_idx_cnn].std(axis=0, keepdims=True)
y_std_lr[y_std_lr < 1e-6] = 1.0
Y_norm_lr = (Y_cnn_np - y_mean_lr) / y_std_lr

X_train_lr = np.column_stack([
    np.ones(len(train_idx_cnn)),
    X_norm_lr[train_idx_cnn]
])

regularizer_lr = np.eye(X_train_lr.shape[1])
regularizer_lr[0, 0] = 0.0

ridge_alpha_lr = 1.0
ridge_weights_lr = np.linalg.solve(
    X_train_lr.T @ X_train_lr + ridge_alpha_lr * regularizer_lr,
    X_train_lr.T @ Y_norm_lr[train_idx_cnn]
)

for name, indices in (
    ("Train", train_idx_cnn),
    ("Validation", val_idx_cnn),
    ("Held-out tail", test_idx_cnn),
):
    X_eval_lr = np.column_stack([
        np.ones(len(indices)),
        X_norm_lr[indices]
    ])
    ridge_pred = (X_eval_lr @ ridge_weights_lr) * y_std_lr + y_mean_lr
    speed_mae, heading_mae = score_cnn(Y_cnn_np[indices], ridge_pred)

    print(
        f"{name}: ridge speed MAE={speed_mae * 3.6:.2f} km/h, "
        f"heading MAE={heading_mae:.1f}°"
    )

Train: ridge speed MAE=9.78 km/h, heading MAE=33.1°
Validation: ridge speed MAE=11.04 km/h, heading MAE=83.5°
Held-out tail: ridge speed MAE=5.43 km/h, heading MAE=33.7°


In [9]:
import math

window_len_gru = 20  # 2 seconds of phone readings
horizon_gru = 10     # about 1 second ahead

X_delta = []
Y_delta = []
actual_horizons = []

for segment in segments_big:
    for end in range(window_len_gru - 1, len(segment) - horizon_gru):
        current = segment[end]
        future = segment[end + horizon_gru]

        h_current = math.atan2(
            current["target_heading_sin"],
            current["target_heading_cos"],
        )
        h_future = math.atan2(
            future["target_heading_sin"],
            future["target_heading_cos"],
        )

        # Shortest signed angle change, wrapped across 0°/360°.
        delta_heading = math.atan2(
            math.sin(h_future - h_current),
            math.cos(h_future - h_current),
        )

        begin = end - window_len_gru + 1
        X_delta.append([
            [segment[k][name] for name in phone_cols_big]
            for k in range(begin, end + 1)
        ])

        Y_delta.append([
            future["target_speed_mps"] - current["target_speed_mps"],
            math.sin(delta_heading),
            math.cos(delta_heading),
        ])

        actual_horizons.append(future["time_s"] - current["time_s"])

delta_heading_deg = [
    math.degrees(math.atan2(y[1], y[2]))
    for y in Y_delta
]

print("Examples:", len(X_delta))
print("Input shape:", (len(X_delta), window_len_gru, len(phone_cols_big)))
print("Target shape:", (len(Y_delta), 3))
print("Actual horizon (s):",
      round(min(actual_horizons), 3), "to",
      round(max(actual_horizons), 3))
print("Speed-change range (m/s):",
      round(min(y[0] for y in Y_delta), 3), "to",
      round(max(y[0] for y in Y_delta), 3))
print("Heading-change range (degrees):",
      round(min(delta_heading_deg), 1), "to",
      round(max(delta_heading_deg), 1))

Examples: 4813
Input shape: (4813, 20, 6)
Target shape: (4813, 3)
Actual horizon (s): 0.991 to 1.009
Speed-change range (m/s): -17.139 to 16.868
Heading-change range (degrees): -176.7 to 169.6


In [10]:
import numpy as np

delta_speed = np.array([y[0] for y in Y_delta])
delta_heading_deg = np.degrees(np.arctan2(
    [y[1] for y in Y_delta],
    [y[2] for y in Y_delta],
))

# Match current speeds to the same delta-target examples.
current_speed = []
for segment in segments_big:
    for end in range(window_len_gru - 1, len(segment) - horizon_gru):
        current_speed.append(segment[end]["target_speed_mps"])
current_speed = np.array(current_speed)

print("Absolute speed-change percentiles (m/s):",
      np.round(np.percentile(np.abs(delta_speed), [50, 90, 95, 99, 100]), 2))
print("Count with |Δspeed| > 5 m/s:", int(np.sum(np.abs(delta_speed) > 5)))
print("Count with |Δspeed| > 10 m/s:", int(np.sum(np.abs(delta_speed) > 10)))

large_turn = np.abs(delta_heading_deg) > 90
print("Count with |Δheading| > 90°:", int(large_turn.sum()))
print("Of those, current speed < 2 m/s:",
      int(np.sum(large_turn & (current_speed < 2))))

Absolute speed-change percentiles (m/s): [ 0.14  0.69  1.01  3.67 17.14]
Count with |Δspeed| > 5 m/s: 39
Count with |Δspeed| > 10 m/s: 35
Count with |Δheading| > 90°: 20
Of those, current speed < 2 m/s: 20


In [11]:
speed_outliers = []

for segment in segments_big:
    for end in range(window_len_gru - 1, len(segment) - horizon_gru):
        current = segment[end]
        future = segment[end + horizon_gru]
        delta_v = future["target_speed_mps"] - current["target_speed_mps"]

        if abs(delta_v) > 10:
            current_v_ms = current["time_ms"] + current["match_error_ms"]
            future_v_ms = future["time_ms"] + future["match_error_ms"]
            v0 = v_by_ms_big[current_v_ms]
            v1 = v_by_ms_big[future_v_ms]

            speed_outliers.append((abs(delta_v), delta_v, current, future, v0, v1))

for _, delta_v, current, future, v0, v1 in sorted(
    speed_outliers, reverse=True, key=lambda item: item[0]
)[:8]:
    print(
        f"{current['time_s']:.3f}→{future['time_s']:.3f}s | "
        f"ΔGPS speed={delta_v:+.1f} m/s | "
        f"GPS={float(v0['Velocity (km/hr)']):.1f}→"
        f"{float(v1['Velocity (km/hr)']):.1f} km/h | "
        f"ECU={v0['Indicated Vehicle Speed (km/hr)']}→"
        f"{v1['Indicated Vehicle Speed (km/hr)']} km/h | "
        f"long accel={v0['Indicated Longitudinal Acceleration (g)']}→"
        f"{v1['Indicated Longitudinal Acceleration (g)']} g | "
        f"satellites={v0['No of GPS Satellites Available']}→"
        f"{v1['No of GPS Satellites Available']}"
    )

29977.206→29978.207s | ΔGPS speed=-17.1 m/s | GPS=61.7→0.0 km/h | ECU=61.66→61.23 km/h | long accel=0.009180069→0.009180069 g | satellites=134.0→0.0
29976.807→29977.806s | ΔGPS speed=-17.1 m/s | GPS=61.7→0.0 km/h | ECU=61.62→61.37 km/h | long accel=0.009180069→0.009180069 g | satellites=135.0→0.0
29977.308→29978.307s | ΔGPS speed=-17.1 m/s | GPS=61.5→0.0 km/h | ECU=61.62→61.32 km/h | long accel=0.009180069→0.009180069 g | satellites=134.0→0.0
29977.006→29978.006s | ΔGPS speed=-17.1 m/s | GPS=61.4→0.0 km/h | ECU=61.61→61.28 km/h | long accel=0.009180069→0.009180069 g | satellites=135.0→0.0
29976.907→29977.906s | ΔGPS speed=-17.0 m/s | GPS=61.1→0.0 km/h | ECU=61.58→61.29 km/h | long accel=0.009180069→0.009180069 g | satellites=135.0→0.0
29977.407→29978.406s | ΔGPS speed=-16.9 m/s | GPS=60.7→0.0 km/h | ECU=61.49→61.27 km/h | long accel=0.009180069→0.009180069 g | satellites=133.0→0.0
29978.106→29979.106s | ΔGPS speed=+16.9 m/s | GPS=0.0→60.7 km/h | ECU=61.17→60.84 km/h | long accel=0.0091

In [12]:
import math
import numpy as np

Y_delta_ecu = []
heading_valid = []
gps_invalid_examples = 0
low_speed_examples = 0

for segment in segments_big:
    for end in range(window_len_gru - 1, len(segment) - horizon_gru):
        current = segment[end]
        future = segment[end + horizon_gru]

        v0_ms = current["time_ms"] + current["match_error_ms"]
        v1_ms = future["time_ms"] + future["match_error_ms"]
        v0 = v_by_ms_big[v0_ms]
        v1 = v_by_ms_big[v1_ms]

        # Continuous vehicle-side speed reference, converted to m/s.
        speed0 = float(v0["Indicated Vehicle Speed (km/hr)"]) / 3.6
        speed1 = float(v1["Indicated Vehicle Speed (km/hr)"]) / 3.6

        gps_ok0 = float(v0["No of GPS Satellites Available"]) > 0
        gps_ok1 = float(v1["No of GPS Satellites Available"]) > 0
        speed_ok = min(speed0, speed1) >= 2.0
        valid_heading = gps_ok0 and gps_ok1 and speed_ok

        if not (gps_ok0 and gps_ok1):
            gps_invalid_examples += 1
        if not speed_ok:
            low_speed_examples += 1

        if valid_heading:
            h0 = math.radians(float(v0["Heading (degrees)"]))
            h1 = math.radians(float(v1["Heading (degrees)"]))
            dh = math.atan2(math.sin(h1 - h0), math.cos(h1 - h0))
            sin_dh, cos_dh = math.sin(dh), math.cos(dh)
        else:
            # Placeholder only; the heading loss will mask this example.
            sin_dh, cos_dh = 0.0, 1.0

        Y_delta_ecu.append([speed1 - speed0, sin_dh, cos_dh])
        heading_valid.append(valid_heading)

assert len(Y_delta_ecu) == len(X_delta)

ecu_delta_speed = np.array([y[0] for y in Y_delta_ecu])

print("Examples:", len(Y_delta_ecu))
print("ECU speed-change range (m/s):",
      round(ecu_delta_speed.min(), 3), "to",
      round(ecu_delta_speed.max(), 3))
print("Absolute ECU speed-change percentiles (m/s):",
      np.round(np.percentile(np.abs(ecu_delta_speed), [50, 90, 95, 99, 100]), 2))
print("Heading labels valid:", sum(heading_valid))
print("Heading labels masked for GPS outage:", gps_invalid_examples)
print("Examples with an endpoint below 2 m/s:", low_speed_examples)

Examples: 4813
ECU speed-change range (m/s): -2.506 to 4.617
Absolute ECU speed-change percentiles (m/s): [0.12 0.56 0.78 2.09 4.62]
Heading labels valid: 4643
Heading labels masked for GPS outage: 58
Examples with an endpoint below 2 m/s: 112


In [13]:
from collections import Counter

counts = Counter()

for segment in segments_big:
    for end in range(window_len_gru - 1, len(segment) - horizon_gru):
        current = segment[end]
        future = segment[end + horizon_gru]

        v0_ms = current["time_ms"] + current["match_error_ms"]
        v1_ms = future["time_ms"] + future["match_error_ms"]
        v0 = v_by_ms_big[v0_ms]
        v1 = v_by_ms_big[v1_ms]

        gps_pair_ok = (
            float(v0["No of GPS Satellites Available"]) > 0
            and float(v1["No of GPS Satellites Available"]) > 0
        )

        speed0 = float(v0["Indicated Vehicle Speed (km/hr)"]) / 3.6
        speed1 = float(v1["Indicated Vehicle Speed (km/hr)"]) / 3.6
        speed_pair_ok = min(speed0, speed1) >= 2.0

        if gps_pair_ok and speed_pair_ok:
            counts["valid_heading"] += 1
        elif not gps_pair_ok and speed_pair_ok:
            counts["gps_outage_only"] += 1
        elif gps_pair_ok and not speed_pair_ok:
            counts["low_speed_only"] += 1
        else:
            counts["both_conditions"] += 1

print("Recomputed categories:", dict(counts))
print("Total:", sum(counts.values()))
print("Existing mask length/valid count:",
      len(heading_valid), sum(heading_valid))
print("Input/target lengths:", len(X_delta), len(Y_delta_ecu))

Recomputed categories: {'valid_heading': 4643, 'low_speed_only': 112, 'gps_outage_only': 58}
Total: 4813
Existing mask length/valid count: 4813 4643
Input/target lengths: 4813 4813


In [14]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

Xg = np.asarray(X_delta, dtype=np.float32)          # (N, time, channels)
Yg = np.asarray(Y_delta_ecu, dtype=np.float32)      # (N, 3)
mask_g = np.asarray(heading_valid, dtype=bool)

n = len(Xg)
purge = window_len_gru + horizon_gru - 1  # 29 windows

train_end_g = int(0.70 * n)
val_end_g = int(0.85 * n)

train_g = np.arange(0, train_end_g)
val_g = np.arange(train_end_g + purge, val_end_g)
test_g = np.arange(val_end_g + purge, n)

print("Train / validation / held-out tail:",
      len(train_g), len(val_g), len(test_g))

# Normalize phone channels using training data only.
x_mean_g = Xg[train_g].mean(axis=(0, 1), keepdims=True)
x_std_g = Xg[train_g].std(axis=(0, 1), keepdims=True)
x_std_g[x_std_g < 1e-6] = 1.0
Xg_norm = (Xg - x_mean_g) / x_std_g

# Normalize speed-change targets; leave circular heading targets as sin/cos.
speed_mean_g = Yg[train_g, 0].mean()
speed_std_g = Yg[train_g, 0].std()
if speed_std_g < 1e-6:
    speed_std_g = 1.0

Yg_norm = Yg.copy()
Yg_norm[:, 0] = (Yg[:, 0] - speed_mean_g) / speed_std_g

Xg_t = torch.from_numpy(Xg_norm)
Yg_t = torch.from_numpy(Yg_norm)
mask_g_t = torch.from_numpy(mask_g)

train_g_t = torch.as_tensor(train_g, dtype=torch.long)
val_g_t = torch.as_tensor(val_g, dtype=torch.long)
test_g_t = torch.as_tensor(test_g, dtype=torch.long)

train_loader_g = DataLoader(
    TensorDataset(
        Xg_t[train_g_t],
        Yg_t[train_g_t],
        mask_g_t[train_g_t],
    ),
    batch_size=128,
    shuffle=True,
)

class DeltaGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(
            input_size=Xg.shape[2],
            hidden_size=32,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(32, 3),
        )

    def forward(self, x):
        sequence, _ = self.gru(x)
        return self.head(sequence[:, -1, :])

def masked_loss(prediction, target, heading_mask):
    speed_loss = F.smooth_l1_loss(
        prediction[:, 0], target[:, 0]
    )

    if heading_mask.any().item():
        heading_loss = F.mse_loss(
            prediction[heading_mask, 1:3],
            target[heading_mask, 1:3],
        )
    else:
        heading_loss = prediction[:, 1:3].sum() * 0.0

    return speed_loss + heading_loss

torch.manual_seed(42)
gru_model = DeltaGRU()
optimizer_g = torch.optim.AdamW(
    gru_model.parameters(), lr=5e-4, weight_decay=1e-4
)

best_val_g = float("inf")
best_state_g = None
best_epoch_g = 0
stale_g = 0

for epoch in range(1, 51):
    gru_model.train()
    train_loss_sum = 0.0
    train_count = 0

    for xb, yb, mb in train_loader_g:
        optimizer_g.zero_grad()
        prediction = gru_model(xb)
        loss = masked_loss(prediction, yb, mb)
        loss.backward()
        optimizer_g.step()

        train_loss_sum += loss.item() * len(xb)
        train_count += len(xb)

    gru_model.eval()
    with torch.no_grad():
        val_prediction = gru_model(Xg_t[val_g_t])
        val_loss_g = masked_loss(
            val_prediction, Yg_t[val_g_t], mask_g_t[val_g_t]
        ).item()

    if val_loss_g < best_val_g - 1e-5:
        best_val_g = val_loss_g
        best_state_g = {
            key: value.detach().clone()
            for key, value in gru_model.state_dict().items()
        }
        best_epoch_g = epoch
        stale_g = 0
    else:
        stale_g += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:2d} | "
            f"train loss {train_loss_sum / train_count:.4f} | "
            f"validation loss {val_loss_g:.4f}"
        )

    if stale_g >= 10:
        break

gru_model.load_state_dict(best_state_g)
print("Best epoch:", best_epoch_g)

def evaluate_delta_gru(name, indices):
    idx_t = torch.as_tensor(indices, dtype=torch.long)

    gru_model.eval()
    with torch.no_grad():
        pred_norm = gru_model(Xg_t[idx_t]).numpy()

    prediction = pred_norm.copy()
    prediction[:, 0] = prediction[:, 0] * speed_std_g + speed_mean_g
    truth = Yg[indices]

    speed_mae = np.mean(np.abs(prediction[:, 0] - truth[:, 0]))
    baseline_delta = Yg[train_g, 0].mean()
    baseline_speed_mae = np.mean(
        np.abs(baseline_delta - truth[:, 0])
    )

    valid = mask_g[indices]
    if valid.any():
        pred_heading = np.degrees(
            np.arctan2(prediction[valid, 1], prediction[valid, 2])
        )
        true_heading = np.degrees(
            np.arctan2(truth[valid, 1], truth[valid, 2])
        )
        heading_error = (pred_heading - true_heading + 180) % 360 - 180
        heading_mae = np.mean(np.abs(heading_error))
        no_turn_mae = np.mean(np.abs(true_heading))
    else:
        heading_mae = float("nan")
        no_turn_mae = float("nan")

    print(
        f"{name}: Δspeed MAE={speed_mae:.3f} m/s "
        f"(mean-change baseline {baseline_speed_mae:.3f}); "
        f"Δheading MAE={heading_mae:.1f}° "
        f"(no-turn baseline {no_turn_mae:.1f}°, "
        f"{int(valid.sum())} valid headings)"
    )

evaluate_delta_gru("Validation", val_g)
evaluate_delta_gru("Held-out tail", test_g)

Train / validation / held-out tail: 3369 693 693
Epoch  1 | train loss 0.6932 | validation loss 0.6378
Epoch  5 | train loss 0.2586 | validation loss 0.3218
Epoch 10 | train loss 0.2555 | validation loss 0.3225
Best epoch: 4
Validation: Δspeed MAE=0.313 m/s (mean-change baseline 0.316); Δheading MAE=3.1° (no-turn baseline 2.4°, 675 valid headings)
Held-out tail: Δspeed MAE=0.157 m/s (mean-change baseline 0.166); Δheading MAE=2.7° (no-turn baseline 2.3°, 693 valid headings)


In [15]:
def heading_degrees(y):
    return np.degrees(np.arctan2(y[:, 1], y[:, 2]))

# Define "larger change" using the training data only.
train_speed_threshold = np.quantile(
    np.abs(Yg[train_g, 0]), 0.75
)

train_valid_headings = mask_g[train_g]
train_heading_threshold = np.quantile(
    np.abs(heading_degrees(Yg[train_g][train_valid_headings])), 0.75
)

def check_change_events(name, indices):
    idx_t = torch.as_tensor(indices, dtype=torch.long)

    gru_model.eval()
    with torch.no_grad():
        pred_norm = gru_model(Xg_t[idx_t]).numpy()

    pred = pred_norm.copy()
    pred[:, 0] = pred[:, 0] * speed_std_g + speed_mean_g
    truth = Yg[indices]

    # Larger speed changes: compare GRU with training-mean Δspeed.
    speed_event = np.abs(truth[:, 0]) >= train_speed_threshold
    speed_baseline = Yg[train_g, 0].mean()

    if speed_event.any():
        gru_speed_mae = np.mean(
            np.abs(pred[speed_event, 0] - truth[speed_event, 0])
        )
        base_speed_mae = np.mean(
            np.abs(speed_baseline - truth[speed_event, 0])
        )
    else:
        gru_speed_mae = base_speed_mae = float("nan")

    # Larger valid heading changes: compare GRU with the no-turn baseline.
    true_heading = heading_degrees(truth)
    pred_heading = heading_degrees(pred)
    heading_event = (
        mask_g[indices]
        & (np.abs(true_heading) >= train_heading_threshold)
    )

    if heading_event.any():
        heading_error = (
            pred_heading[heading_event] - true_heading[heading_event] + 180
        ) % 360 - 180
        no_turn_error = (
            -true_heading[heading_event] + 180
        ) % 360 - 180

        gru_heading_mae = np.mean(np.abs(heading_error))
        no_turn_mae = np.mean(np.abs(no_turn_error))
    else:
        gru_heading_mae = no_turn_mae = float("nan")

    print(
        f"{name} | speed events (|Δv| ≥ {train_speed_threshold:.3f} m/s): "
        f"n={speed_event.sum()}, GRU={gru_speed_mae:.3f}, "
        f"mean baseline={base_speed_mae:.3f} m/s"
    )
    print(
        f"{name} | heading events (|Δheading| ≥ "
        f"{train_heading_threshold:.2f}°): "
        f"n={heading_event.sum()}, GRU={gru_heading_mae:.2f}, "
        f"no-turn baseline={no_turn_mae:.2f}°"
    )

check_change_events("Validation", val_g)
check_change_events("Held-out tail", test_g)

Validation | speed events (|Δv| ≥ 0.217 m/s): n=379, GRU=0.486, mean baseline=0.490 m/s
Validation | heading events (|Δheading| ≥ 36.40°): n=0, GRU=nan, no-turn baseline=nan°
Held-out tail | speed events (|Δv| ≥ 0.217 m/s): n=160, GRU=0.483, mean baseline=0.487 m/s
Held-out tail | heading events (|Δheading| ≥ 36.40°): n=0, GRU=nan, no-turn baseline=nan°


In [16]:
def summarize_heading_split(name, indices):
    split_mask = mask_g[indices]
    y = Yg[indices][split_mask]

    heading_deg = np.degrees(np.arctan2(y[:, 1], y[:, 2]))
    abs_heading = np.abs(heading_deg)

    print(f"\n{name}: {len(heading_deg)} valid headings")
    print(
        "Absolute Δheading percentiles "
        "[0, 25, 50, 75, 90, 95, 99, 100]:",
        np.round(
            np.percentile(abs_heading, [0, 25, 50, 75, 90, 95, 99, 100]),
            1,
        ),
    )
    print(
        "Counts above [5, 10, 20, 30, 45, 90] degrees:",
        {
            threshold: int(np.sum(abs_heading > threshold))
            for threshold in [5, 10, 20, 30, 45, 90]
        },
    )

summarize_heading_split("Train", train_g)
summarize_heading_split("Validation", val_g)
summarize_heading_split("Held-out tail", test_g)


Train: 3217 valid headings
Absolute Δheading percentiles [0, 25, 50, 75, 90, 95, 99, 100]: [ 0.   4.1 33.7 36.4 45.3 46.4 47.5 53. ]
Counts above [5, 10, 20, 30, 45, 90] degrees: {5: 2316, 10: 2085, 20: 1905, 30: 1845, 45: 367, 90: 0}

Validation: 675 valid headings
Absolute Δheading percentiles [0, 25, 50, 75, 90, 95, 99, 100]: [ 0.   0.7  1.4  2.9  5.9  8.6 10.5 11.8]
Counts above [5, 10, 20, 30, 45, 90] degrees: {5: 92, 10: 14, 20: 0, 30: 0, 45: 0, 90: 0}

Held-out tail: 693 valid headings
Absolute Δheading percentiles [0, 25, 50, 75, 90, 95, 99, 100]: [ 0.   0.9  1.8  3.2  5.4  6.7  7.7 10. ]
Counts above [5, 10, 20, 30, 45, 90] degrees: {5: 87, 10: 0, 20: 0, 30: 0, 45: 0, 90: 0}


In [18]:
terms = ("v_", "vm", "vehicle", "aligned", "rows", "prefix")

for name, obj in list(globals().items()):
    if any(term in name.lower() for term in terms):
        shape = getattr(obj, "shape", None)
        try:
            length = len(obj)
        except TypeError:
            length = None
        print(f"{name}: type={type(obj).__name__}, shape={shape}, len={length}")

V_URL: type=str, shape=None, len=162
read_prefix_csv: type=function, shape=None, len=None
v_big: type=list, shape=None, len=4884
v_tod: type=list, shape=None, len=4884
v_by_ms_big: type=dict, shape=None, len=4883
v_times_ms_big: type=list, shape=None, len=4883
aligned_big: type=list, shape=None, len=4871
outside_v_window: type=int, shape=None, len=None
bad_rows: type=int, shape=None, len=None
v_time_ms: type=int, shape=None, len=None
v_row: type=dict, shape=None, len=29
current_v_ms: type=int, shape=None, len=None
future_v_ms: type=int, shape=None, len=None


In [19]:
print("V-M columns:")
print(list(v_big[0].keys()))

print("\nCandidate fields and example values:")
for key, value in v_big[0].items():
    if any(term in key.lower() for term in
           ("acceler", "brak", "speed", "time")):
        print(repr(key), "->", repr(value))

V-M columns:
['No of GPS Satellites Available', 'Time Since Start of Day (seconds)', 'Latitude (degrees)', 'Longitude (degrees)', 'Velocity (km/hr)', 'Heading (degrees)', 'Height (km)', 'Vertical velocity (km/hr)', 'Sample period (seconds)', 'Steering Angle (degrees)', 'Wheel Speed Front Left (rad/sec)', 'Wheel Speed Front Right (rad/sec)', 'Wheel Speed Rear Left (rad/sec)', 'Wheel Speed Rear Right (rad/sec)', 'Yaw Rate (deg/sec)', 'Indicated Vehicle Speed (km/hr)', 'Indicated Longitudinal Acceleration (g)', 'Indicated Lateral Acceleration (g)', 'Handbrake (0 or 1)', 'Gear Requested (Number fof gear employed 1-5)', 'Gear (Number fof gear employed 1-5)', 'Engine Speed (rev/min)', 'Coolant Temperature (degrees)', 'Clutch Position (0 or 1)', 'Brake Pressure (psi)', 'Brake Position (0 or 1)', 'Battery Voltage (volts)', 'Air Temperature (degrees)', 'Accelerator Pedal Position (0 or 1)']

Candidate fields and example values:
'Time Since Start of Day (seconds)' -> '29608.4'
'Wheel Speed Front

In [20]:
import numpy as np

def vm_numeric_column(name):
    values = []
    for row in v_big:
        try:
            values.append(float(row[name]))
        except (KeyError, TypeError, ValueError):
            values.append(np.nan)
    return np.asarray(values, dtype=float)

acc_g = vm_numeric_column("Indicated Longitudinal Acceleration (g)")
brake_position = vm_numeric_column("Brake Position (0 or 1)")
brake_pressure = vm_numeric_column("Brake Pressure (psi)")

print("Rows:", len(v_big))
print(
    "Longitudinal acceleration percentiles [min, 1, 5, 50, 95, 99, max]:",
    np.round(np.nanpercentile(acc_g, [0, 1, 5, 50, 95, 99, 100]), 3),
)

for threshold in [0.1, 0.2, 0.3]:
    hard_decel = np.isfinite(acc_g) & (acc_g <= -threshold)
    print(
        f"Acceleration <= -{threshold:.1f} g: "
        f"{hard_decel.sum()} samples; "
        f"with Brake Position > 0: "
        f"{np.sum(hard_decel & (brake_position > 0))}"
    )

for name, values in [
    ("Brake Position", brake_position),
    ("Brake Pressure (psi)", brake_pressure),
]:
    finite = values[np.isfinite(values)]
    rounded, counts = np.unique(np.round(finite, 3), return_counts=True)
    most_common = sorted(
        zip(rounded, counts), key=lambda item: item[1], reverse=True
    )[:12]
    print(f"\n{name}: min={finite.min():.3f}, max={finite.max():.3f}")
    print("Most common values (rounded):", most_common)

Rows: 4884
Longitudinal acceleration percentiles [min, 1, 5, 50, 95, 99, max]: [-0.37  -0.171 -0.13  -0.035  0.069  0.18   0.44 ]
Acceleration <= -0.1 g: 466 samples; with Brake Position > 0: 102
Acceleration <= -0.2 g: 25 samples; with Brake Position > 0: 18
Acceleration <= -0.3 g: 8 samples; with Brake Position > 0: 4

Brake Position: min=0.000, max=1.000
Most common values (rounded): [(np.float64(0.0), np.int64(4493)), (np.float64(1.0), np.int64(391))]

Brake Pressure (psi): min=-1.400, max=25.680
Most common values (rounded): [(np.float64(0.03), np.int64(1957)), (np.float64(0.02), np.int64(706)), (np.float64(-0.2), np.int64(443)), (np.float64(0.05), np.int64(315)), (np.float64(-0.21), np.int64(268)), (np.float64(0.04), np.int64(248)), (np.float64(0.01), np.int64(189)), (np.float64(-0.18), np.int64(128)), (np.float64(-0.19), np.int64(114)), (np.float64(-0.22), np.int64(91)), (np.float64(0.27), np.int64(18)), (np.float64(2.65), np.int64(15))]


In [21]:
yaw_rate = vm_numeric_column("Yaw Rate (deg/sec)")

print(
    "Yaw-rate percentiles [min, 1, 5, 25, 50, 75, 95, 99, max]:",
    np.round(
        np.nanpercentile(yaw_rate, [0, 1, 5, 25, 50, 75, 95, 99, 100]),
        2,
    ),
)

for threshold in [1, 2, 5, 10, 20]:
    valid = np.isfinite(yaw_rate)
    print(
        f"|yaw rate| > {threshold:2d} deg/s: "
        f"{np.sum(valid & (yaw_rate > threshold))} positive, "
        f"{np.sum(valid & (yaw_rate < -threshold))} negative"
    )

Yaw-rate percentiles [min, 1, 5, 25, 50, 75, 95, 99, max]: [-51.   -47.1  -45.7  -34.8   -2.7    0.     7.1   16.42  27.1 ]
|yaw rate| >  1 deg/s: 801 positive, 2884 negative
|yaw rate| >  2 deg/s: 665 positive, 2560 negative
|yaw rate| >  5 deg/s: 383 positive, 2252 negative
|yaw rate| > 10 deg/s: 161 positive, 1983 negative
|yaw rate| > 20 deg/s: 33 positive, 1903 negative


In [22]:
vehicle_speed = vm_numeric_column("Indicated Vehicle Speed (km/hr)")
block_size = 250  # roughly 25 seconds at 10 Hz

for start in range(0, len(yaw_rate), block_size):
    stop = min(start + block_size, len(yaw_rate))
    y = yaw_rate[start:stop]
    s = vehicle_speed[start:stop]

    valid_y = np.isfinite(y)
    valid_s = np.isfinite(s)
    y = y[valid_y]
    s = s[valid_s]

    if len(y) == 0:
        continue

    print(
        f"{start:4d}:{stop:4d} | "
        f"speed median={np.median(s):5.1f} km/h | "
        f"yaw p5/median/p95="
        f"{np.percentile(y, 5):6.1f}/"
        f"{np.median(y):6.1f}/"
        f"{np.percentile(y, 95):6.1f} deg/s | "
        f"<-5={np.sum(y < -5):3d}, >5={np.sum(y > 5):3d}"
    )

   0: 250 | speed median= 25.8 km/h | yaw p5/median/p95= -23.8/  -4.6/  10.5 deg/s | <-5=118, >5= 53
 250: 500 | speed median= 27.2 km/h | yaw p5/median/p95= -47.6/ -45.6/ -42.9 deg/s | <-5=250, >5=  0
 500: 750 | speed median= 28.9 km/h | yaw p5/median/p95= -47.2/ -45.5/ -43.5 deg/s | <-5=250, >5=  0
 750:1000 | speed median= 22.9 km/h | yaw p5/median/p95= -36.9/ -35.8/ -33.1 deg/s | <-5=250, >5=  0
1000:1250 | speed median= 24.5 km/h | yaw p5/median/p95= -38.5/ -36.4/ -35.1 deg/s | <-5=250, >5=  0
1250:1500 | speed median= 22.5 km/h | yaw p5/median/p95= -35.0/ -34.0/ -32.5 deg/s | <-5=250, >5=  0
1500:1750 | speed median= 23.4 km/h | yaw p5/median/p95= -36.6/ -35.2/ -34.0 deg/s | <-5=250, >5=  0
1750:2000 | speed median= 22.2 km/h | yaw p5/median/p95= -36.6/ -33.9/ -30.6 deg/s | <-5=250, >5=  0
2000:2250 | speed median= 19.1 km/h | yaw p5/median/p95= -34.9/  -2.4/   1.0 deg/s | <-5=106, >5=  6
2250:2500 | speed median= 33.1 km/h | yaw p5/median/p95= -41.5/   2.1/  24.5 deg/s | <-5= 8

In [23]:
speed_kmh = vm_numeric_column("Indicated Vehicle Speed (km/hr)")
print(
    "ECU speed [min, p25, median, p75, max] km/h:",
    np.round(np.nanpercentile(speed_kmh, [0, 25, 50, 75, 100]), 1),
)

sample = aligned_big[0]
print("\naligned_big entry type:", type(sample).__name__)

if isinstance(sample, dict):
    print("keys:", list(sample.keys()))
elif isinstance(sample, (tuple, list)):
    print("length:", len(sample))
    for i, item in enumerate(sample):
        detail = list(item.keys()) if isinstance(item, dict) else type(item).__name__
        print(i, detail)

ECU speed [min, p25, median, p75, max] km/h: [ 0.  23.4 32.8 49.  67.9]

aligned_big entry type: dict
keys: ['time_ms', 'time_s', 'match_error_ms', 'ACCELEROMETER X (m/s�)', 'ACCELEROMETER Y (m/s�)', 'ACCELEROMETER Z (m/s�)', 'GYROSCOPE Yaw (rad/s)', 'GYROSCOPE Pitch (rad/s)', 'GYROSCOPE Roll (rad/s)', 'target_speed_mps', 'target_heading_sin', 'target_heading_cos']


In [24]:
v_times = np.asarray(v_times_ms_big, dtype=np.int64)
target_check, gps_check, ecu_check, time_error_check = [], [], [], []

for aligned_row in aligned_big:
    t_ms = int(round(float(aligned_row["time_ms"])))
    pos = int(np.searchsorted(v_times, t_ms))
    candidates = [j for j in (pos - 1, pos) if 0 <= j < len(v_times)]
    j = min(candidates, key=lambda k: abs(int(v_times[k]) - t_ms))

    v_row_check = v_by_ms_big[int(v_times[j])]

    target_check.append(float(aligned_row["target_speed_mps"]))
    gps_check.append(float(v_row_check["Velocity (km/hr)"]) / 3.6)
    ecu_check.append(
        float(v_row_check["Indicated Vehicle Speed (km/hr)"]) / 3.6
    )
    time_error_check.append(abs(int(v_times[j]) - t_ms))

target_check = np.asarray(target_check)
gps_check = np.asarray(gps_check)
ecu_check = np.asarray(ecu_check)

print(
    "Nearest V-M timestamp error p95/max (ms):",
    np.percentile(time_error_check, 95),
    max(time_error_check),
)
print(
    "Stored target vs GPS Velocity MAE (m/s):",
    np.mean(np.abs(target_check - gps_check)),
)
print(
    "Stored target vs ECU-indicated speed MAE (m/s):",
    np.mean(np.abs(target_check - ecu_check)),
)

Nearest V-M timestamp error p95/max (ms): 7.0 15
Stored target vs GPS Velocity MAE (m/s): 0.0
Stored target vs ECU-indicated speed MAE (m/s): 0.3872783833458792


In [25]:
print("X_delta:", np.asarray(X_delta).shape)
print("Y_delta_ecu:", np.asarray(Y_delta_ecu).shape)
print("segments_big:", type(segments_big).__name__, "len:", len(segments_big))

for i, segment in enumerate(segments_big):
    try:
        n = len(segment)
    except TypeError:
        n = None

    print(
        f"segment {i}: type={type(segment).__name__}, "
        f"len={n}, shape={getattr(segment, 'shape', None)}"
    )

    if n:
        first = segment[0]
        print(
            "  first item:",
            type(first).__name__,
            list(first.keys()) if isinstance(first, dict) else "",
        )

X_delta: (4813, 20, 6)
Y_delta_ecu: (4813, 3)
segments_big: list len: 2
segment 0: type=list, len=210, shape=None
  first item: dict ['time_ms', 'time_s', 'match_error_ms', 'ACCELEROMETER X (m/s�)', 'ACCELEROMETER Y (m/s�)', 'ACCELEROMETER Z (m/s�)', 'GYROSCOPE Yaw (rad/s)', 'GYROSCOPE Pitch (rad/s)', 'GYROSCOPE Roll (rad/s)', 'target_speed_mps', 'target_heading_sin', 'target_heading_cos']
segment 1: type=list, len=4661, shape=None
  first item: dict ['time_ms', 'time_s', 'match_error_ms', 'ACCELEROMETER X (m/s�)', 'ACCELEROMETER Y (m/s�)', 'ACCELEROMETER Z (m/s�)', 'GYROSCOPE Yaw (rad/s)', 'GYROSCOPE Pitch (rad/s)', 'GYROSCOPE Roll (rad/s)', 'target_speed_mps', 'target_heading_sin', 'target_heading_cos']


In [26]:
W, H = 20, 10
v_times = np.asarray(v_times_ms_big, dtype=np.int64)

def nearest_v_row(time_ms):
    pos = int(np.searchsorted(v_times, int(round(float(time_ms)))))
    candidates = [j for j in (pos - 1, pos) if 0 <= j < len(v_times)]
    j = min(candidates, key=lambda k: abs(int(v_times[k]) - int(round(float(time_ms)))))
    return v_by_ms_big[int(v_times[j])], abs(int(v_times[j]) - int(round(float(time_ms))))

X_speed_check = []
speed_future_ecu = []
delta_speed_check = []
max_time_error_ms = 0

for segment in segments_big:
    segment_speed_mps = []

    for row in segment:
        v_row, time_error = nearest_v_row(row["time_ms"])
        max_time_error_ms = max(max_time_error_ms, time_error)
        segment_speed_mps.append(
            float(v_row["Indicated Vehicle Speed (km/hr)"]) / 3.6
        )

    for start in range(len(segment) - W - H + 1):
        last_input = start + W - 1
        future = last_input + H

        X_speed_check.append([
            [float(row[col]) for col in phone_cols_big]
            for row in segment[start:start + W]
        ])
        speed_future_ecu.append(segment_speed_mps[future])
        delta_speed_check.append(
            segment_speed_mps[future] - segment_speed_mps[last_input]
        )

X_speed_check = np.asarray(X_speed_check, dtype=np.float32)
speed_future_ecu = np.asarray(speed_future_ecu, dtype=np.float32)
delta_speed_check = np.asarray(delta_speed_check, dtype=np.float32)

print("Rebuilt phone windows:", X_speed_check.shape)
print("Future ECU-speed targets:", speed_future_ecu.shape)
print("Maximum V-M timestamp match error (ms):", max_time_error_ms)
print(
    "Max phone-window difference vs X_delta:",
    np.max(np.abs(X_speed_check - np.asarray(X_delta))),
)
print(
    "MAE of rebuilt ECU Δspeed vs Y_delta_ecu:",
    np.mean(np.abs(delta_speed_check - np.asarray(Y_delta_ecu)[:, 0])),
)
print(
    "Future ECU-speed range (m/s):",
    np.round([speed_future_ecu.min(), speed_future_ecu.max()], 3),
)

Rebuilt phone windows: (4813, 20, 6)
Future ECU-speed targets: (4813,)
Maximum V-M timestamp match error (ms): 15
Max phone-window difference vs X_delta: 4.760742182696731e-07
MAE of rebuilt ECU Δspeed vs Y_delta_ecu: 4.996733855498019e-09
Future ECU-speed range (m/s): [ 0.    18.872]


In [27]:
n = len(X_speed_check)
purge = 29

train_end = int(0.70 * n)
val_end = int(0.85 * n)

train_idx = np.arange(0, train_end)
val_idx = np.arange(train_end + purge, val_end)
test_idx = np.arange(val_end + purge, n)

# Normalize phone channels using training data only.
x_mean = X_speed_check[train_idx].mean(axis=(0, 1), keepdims=True)
x_std = X_speed_check[train_idx].std(axis=(0, 1), keepdims=True)
x_std[x_std < 1e-6] = 1.0

X_norm = (X_speed_check - x_mean) / x_std
X_flat = X_norm.reshape(n, -1).astype(np.float64)
y_speed = speed_future_ecu.astype(np.float64)

# Add an intercept; don't penalize it.
A_train = np.column_stack([np.ones(len(train_idx)), X_flat[train_idx]])
A_val = np.column_stack([np.ones(len(val_idx)), X_flat[val_idx]])
A_test = np.column_stack([np.ones(len(test_idx)), X_flat[test_idx]])

penalty = np.eye(A_train.shape[1])
penalty[0, 0] = 0.0

alpha_scores = []
for alpha in [0.1, 1.0, 10.0, 100.0]:
    coef = np.linalg.solve(
        A_train.T @ A_train + alpha * penalty,
        A_train.T @ y_speed[train_idx],
    )
    val_pred = A_val @ coef
    val_mae = np.mean(np.abs(val_pred - y_speed[val_idx]))
    alpha_scores.append((val_mae, alpha, coef))
    print(f"alpha={alpha:5.1f} | validation MAE={val_mae:.3f} m/s")

_, best_alpha, best_coef = min(alpha_scores, key=lambda item: item[0])

def report_speed(name, indices, design_matrix):
    prediction = design_matrix @ best_coef
    truth = y_speed[indices]
    baseline = np.full(len(indices), y_speed[train_idx].mean())

    model_mae = np.mean(np.abs(prediction - truth))
    baseline_mae = np.mean(np.abs(baseline - truth))

    print(
        f"{name}: Ridge MAE={model_mae:.3f} m/s "
        f"({model_mae * 3.6:.2f} km/h); "
        f"training-mean baseline={baseline_mae:.3f} m/s "
        f"({baseline_mae * 3.6:.2f} km/h)"
    )

print("Chosen alpha:", best_alpha)
report_speed("Validation", val_idx, A_val)
report_speed("Held-out tail", test_idx, A_test)

alpha=  0.1 | validation MAE=2.667 m/s
alpha=  1.0 | validation MAE=2.667 m/s
alpha= 10.0 | validation MAE=2.667 m/s
alpha=100.0 | validation MAE=2.669 m/s
Chosen alpha: 0.1
Validation: Ridge MAE=2.667 m/s (9.60 km/h); training-mean baseline=4.847 m/s (17.45 km/h)
Held-out tail: Ridge MAE=1.752 m/s (6.31 km/h); training-mean baseline=2.325 m/s (8.37 km/h)


In [28]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

# X_norm was normalized using training data only in the Ridge cell.
X_speed_t = torch.as_tensor(X_norm, dtype=torch.float32)

y_mean_gru = y_speed[train_idx].mean()
y_std_gru = y_speed[train_idx].std()
if y_std_gru < 1e-6:
    y_std_gru = 1.0

y_norm_gru = ((y_speed - y_mean_gru) / y_std_gru).astype(np.float32)
y_speed_t = torch.from_numpy(y_norm_gru).unsqueeze(1)

train_idx_t = torch.as_tensor(train_idx, dtype=torch.long)
val_idx_t = torch.as_tensor(val_idx, dtype=torch.long)
test_idx_t = torch.as_tensor(test_idx, dtype=torch.long)

train_loader_speed = DataLoader(
    TensorDataset(X_speed_t[train_idx_t], y_speed_t[train_idx_t]),
    batch_size=128,
    shuffle=True,
)

class SpeedGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(
            input_size=X_speed_check.shape[2],
            hidden_size=32,
            batch_first=True,
        )
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(32, 1))

    def forward(self, x):
        sequence, _ = self.gru(x)
        return self.head(sequence[:, -1, :])

speed_gru = SpeedGRU()
optimizer_speed = torch.optim.AdamW(
    speed_gru.parameters(), lr=5e-4, weight_decay=1e-4
)

best_val_mae = float("inf")
best_speed_state = None
best_speed_epoch = 0
stale = 0

for epoch in range(1, 51):
    speed_gru.train()
    total_loss = 0.0
    total_n = 0

    for xb, yb in train_loader_speed:
        optimizer_speed.zero_grad()
        pred = speed_gru(xb)
        loss = F.smooth_l1_loss(pred, yb)
        loss.backward()
        optimizer_speed.step()
        total_loss += loss.item() * len(xb)
        total_n += len(xb)

    speed_gru.eval()
    with torch.no_grad():
        val_pred_norm = speed_gru(X_speed_t[val_idx_t]).squeeze(1).numpy()
    val_pred = val_pred_norm * y_std_gru + y_mean_gru
    val_mae = np.mean(np.abs(val_pred - y_speed[val_idx]))

    if val_mae < best_val_mae - 1e-4:
        best_val_mae = val_mae
        best_speed_state = {
            k: v.detach().clone() for k, v in speed_gru.state_dict().items()
        }
        best_speed_epoch = epoch
        stale = 0
    else:
        stale += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:2d} | "
            f"train Huber={total_loss / total_n:.4f} | "
            f"validation MAE={val_mae:.3f} m/s"
        )

    if stale >= 8:
        break

speed_gru.load_state_dict(best_speed_state)
print("Best epoch:", best_speed_epoch)

def evaluate_speed_gru(name, indices, index_tensor):
    speed_gru.eval()
    with torch.no_grad():
        pred_norm = speed_gru(X_speed_t[index_tensor]).squeeze(1).numpy()

    pred = pred_norm * y_std_gru + y_mean_gru
    truth = y_speed[indices]
    baseline = np.full(len(indices), y_speed[train_idx].mean())

    model_mae = np.mean(np.abs(pred - truth))
    baseline_mae = np.mean(np.abs(baseline - truth))

    print(
        f"{name}: GRU MAE={model_mae:.3f} m/s "
        f"({model_mae * 3.6:.2f} km/h); "
        f"training-mean baseline={baseline_mae:.3f} m/s "
        f"({baseline_mae * 3.6:.2f} km/h)"
    )

evaluate_speed_gru("Validation", val_idx, val_idx_t)
evaluate_speed_gru("Held-out tail", test_idx, test_idx_t)

Epoch  1 | train Huber=0.4449 | validation MAE=5.005 m/s
Epoch  5 | train Huber=0.2679 | validation MAE=2.125 m/s
Epoch 10 | train Huber=0.2075 | validation MAE=2.755 m/s
Epoch 15 | train Huber=0.1170 | validation MAE=3.137 m/s
Best epoch: 8
Validation: GRU MAE=2.005 m/s (7.22 km/h); training-mean baseline=4.847 m/s (17.45 km/h)
Held-out tail: GRU MAE=2.638 m/s (9.50 km/h); training-mean baseline=2.325 m/s (8.37 km/h)


In [29]:
for name, indices in [
    ("Train", train_idx),
    ("Validation", val_idx),
    ("Held-out tail", test_idx),
]:
    speeds_kmh = y_speed[indices] * 3.6
    print(
        f"{name}: n={len(indices)}, "
        f"mean={speeds_kmh.mean():.1f}, "
        f"p10/median/p90="
        f"{np.percentile(speeds_kmh, 10):.1f}/"
        f"{np.median(speeds_kmh):.1f}/"
        f"{np.percentile(speeds_kmh, 90):.1f}, "
        f"range={speeds_kmh.min():.1f}–{speeds_kmh.max():.1f} km/h"
    )

Train: n=3369, mean=33.9, p10/median/p90=20.8/25.5/64.1, range=0.0–67.9 km/h
Validation: n=693, mean=51.2, p10/median/p90=39.6/52.5/60.8, range=30.9–61.7 km/h
Held-out tail: n=693, mean=41.5, p10/median/p90=32.9/44.2/45.4, range=28.7–46.3 km/h


In [30]:
import inspect
print(inspect.signature(read_prefix_csv))
print("URL variables:", [
    name for name in globals()
    if name.lower().endswith("_url")
])

(file_url, n_bytes)
URL variables: ['S_URL', 'V_URL']


In [31]:
import csv
import io
from urllib.request import Request, urlopen

def remote_size(url):
    req = Request(url, headers={"Range": "bytes=0-0"})
    with urlopen(req) as response:
        content_range = response.headers.get("Content-Range")
        if response.status != 206 or not content_range:
            raise RuntimeError(
                f"Range check failed: status={response.status}, "
                f"Content-Range={content_range}"
            )
        return int(content_range.rsplit("/", 1)[1])

def read_csv_byte_range(url, start, n_bytes, fieldnames):
    end = start + n_bytes - 1
    req = Request(url, headers={"Range": f"bytes={start}-{end}"})

    with urlopen(req) as response:
        content_range = response.headers.get("Content-Range")
        if response.status != 206:
            raise RuntimeError(
                f"Expected HTTP 206, got {response.status} for {content_range}"
            )
        raw = response.read()

    # Discard any partial first/last CSV records.
    first_newline = raw.find(b"\n")
    if first_newline < 0:
        return [], content_range
    raw = raw[first_newline + 1:]

    last_newline = raw.rfind(b"\n")
    if last_newline < 0:
        return [], content_range
    raw = raw[:last_newline + 1]

    text = raw.decode("utf-8-sig", errors="replace")
    rows = list(csv.DictReader(
        io.StringIO(text, newline=""),
        fieldnames=fieldnames,
        skipinitialspace=True,
    ))
    return rows, content_range

# Reuse the V-M header already parsed; fetch only a tiny S-M prefix for its header.
s_header_rows = read_prefix_csv(S_URL, 4096)
s_fields = list(s_header_rows[0].keys())
v_fields = list(v_big[0].keys())

size_s = remote_size(S_URL)
size_v = remote_size(V_URL)
chunk_bytes = 1_048_576

print("File sizes:", size_s, size_v)
print("S-M time-like columns:",
      [k for k in s_fields if "time" in k.lower()])
print("V-M time-like columns:",
      [k for k in v_fields if "time" in k.lower()])

sampled_blocks = []

for fraction in [0.25, 0.50, 0.75]:
    start_s = int(fraction * size_s)
    start_v = int(fraction * size_v)

    s_rows, s_range = read_csv_byte_range(
        S_URL, start_s, chunk_bytes, s_fields
    )
    v_rows, v_range = read_csv_byte_range(
        V_URL, start_v, chunk_bytes, v_fields
    )

    sampled_blocks.append({
        "fraction": fraction,
        "S": s_rows,
        "V": v_rows,
    })

    print(
        f"{fraction:.0%}: "
        f"S-M {len(s_rows)} rows ({s_range}); "
        f"V-M {len(v_rows)} rows ({v_range})"
    )

HTTP: 206 | range: bytes 0-4095/19798721 | bytes: 4096
File sizes: 19798721 22507823
S-M time-like columns: ['TIME SINCE START (ms)']
V-M time-like columns: ['Time Since Start of Day (seconds)']
25%: S-M 5624 rows (bytes 4949680-5998255/19798721); V-M 4832 rows (bytes 5626955-6675530/22507823)
50%: S-M 5572 rows (bytes 9899360-10947935/19798721); V-M 4904 rows (bytes 11253911-12302486/22507823)
75%: S-M 5592 rows (bytes 14849040-15897615/19798721); V-M 4862 rows (bytes 16880867-17929442/22507823)


In [32]:
def nearest_time_errors(query_times, reference_times):
    reference_times = np.sort(np.asarray(reference_times, dtype=float))
    errors = []

    for t in query_times:
        pos = int(np.searchsorted(reference_times, t))
        candidates = [
            j for j in (pos - 1, pos)
            if 0 <= j < len(reference_times)
        ]
        errors.append(min(abs(reference_times[j] - t) for j in candidates))

    return np.asarray(errors)

for block in sampled_blocks:
    s_raw_sec = np.asarray([
        float(row["TIME SINCE START (ms)"]) / 1000
        for row in block["S"]
    ])
    v_sec = np.asarray([
        float(row["Time Since Start of Day (seconds)"])
        for row in block["V"]
    ])

    # Preserve the previously validated S-M timezone correction.
    s_sec = s_raw_sec - 3600

    overlap_sec = max(
        0.0,
        min(s_sec.max(), v_sec.max()) - max(s_sec.min(), v_sec.min())
    )
    errors = nearest_time_errors(s_sec, v_sec)

    print(
        f'{block["fraction"]:.0%}: '
        f'S-M corrected range={s_sec.min():.1f}–{s_sec.max():.1f}s; '
        f'V-M range={v_sec.min():.1f}–{v_sec.max():.1f}s; '
        f'overlap={overlap_sec:.1f}s; '
        f'S rows within 0.1s of V-M={np.sum(errors <= 0.1)}/{len(errors)}'
    )

25%: S-M corrected range=-936.5–-374.2s; V-M range=32228.1–32711.2s; overlap=0.0s; S rows within 0.1s of V-M=0/5624
50%: S-M corrected range=-2710.3–-2151.9s; V-M range=34879.2–35369.5s; overlap=0.0s; S rows within 0.1s of V-M=0/5572
75%: S-M corrected range=-64.0–495.1s; V-M range=37553.3–38039.4s; overlap=0.0s; S rows within 0.1s of V-M=0/5592


In [33]:
s_time_key = "TIME SINCE START (ms)"

for block in sampled_blocks:
    s_sec = np.asarray([
        float(row[s_time_key]) / 1000
        for row in block["S"]
    ])
    v_sec = np.asarray([
        float(row["Time Since Start of Day (seconds)"])
        for row in block["V"]
    ])

    ds = np.diff(s_sec)
    print(
        f'{block["fraction"]:.0%}: '
        f'S raw first/last={s_sec[0]:.1f}/{s_sec[-1]:.1f}s; '
        f'S time decreases within block={np.sum(ds < 0)}; '
        f'median S step={np.median(ds):.4f}s; '
        f'median V−S offset={np.median(v_sec) - np.median(s_sec):.1f}s'
    )

print("\nIn-memory lists containing the raw S-M time column:")
for name, obj in list(globals().items()):
    if (
        isinstance(obj, list)
        and obj
        and isinstance(obj[0], dict)
        and s_time_key in obj[0]
    ):
        print(
            name,
            "rows=", len(obj),
            "first time=", obj[0][s_time_key],
            "last time=", obj[-1][s_time_key],
        )

25%: S raw first/last=2663.5/3225.8s; S time decreases within block=0; median S step=0.1000s; median V−S offset=29525.0s
50%: S raw first/last=889.7/1448.1s; S time decreases within block=0; median S step=0.1000s; median V−S offset=33954.8s
75%: S raw first/last=3536.0/4095.1s; S time decreases within block=0; median S step=0.1000s; median V−S offset=33980.8s

In-memory lists containing the raw S-M time column:
s_big rows= 5642 first time= 4227 last time= 568327
s_header_rows rows= 19 first time= 4227 last time= 6027
s_rows rows= 5592 first time= 3535976 last time= 4095076


In [34]:
s_time_key = "TIME SINCE START (ms)"
v_times_ms = np.asarray(v_times_ms_big, dtype=float)

# Candidate offset: align the first S-M elapsed timestamp to the first V-M time.
offset_ms = (
    float(v_times_ms[0])
    - float(s_big[0][s_time_key])
)
s_prefix_ms = np.asarray(
    [float(row[s_time_key]) + offset_ms for row in s_big]
)

in_v_range = (
    (s_prefix_ms >= v_times_ms.min())
    & (s_prefix_ms <= v_times_ms.max())
)

def nearest_ms_errors(query_ms, reference_ms):
    reference_ms = np.sort(np.asarray(reference_ms, dtype=float))
    errors = []
    for t in query_ms:
        pos = int(np.searchsorted(reference_ms, t))
        candidates = [
            j for j in (pos - 1, pos)
            if 0 <= j < len(reference_ms)
        ]
        errors.append(min(abs(reference_ms[j] - t) for j in candidates))
    return np.asarray(errors)

prefix_errors_ms = nearest_ms_errors(
    s_prefix_ms[in_v_range],
    v_times_ms,
)

print(f"Candidate S-to-V offset: {offset_ms / 1000:.3f} seconds")
print(
    "Original-prefix rows inside V time range:",
    int(in_v_range.sum()),
)
print(
    "Match error p95/max:",
    round(float(np.percentile(prefix_errors_ms, 95)), 1),
    round(float(prefix_errors_ms.max()), 1),
    "ms",
)
print(
    "Within 100 ms:",
    int(np.sum(prefix_errors_ms <= 100)),
    "/",
    len(prefix_errors_ms),
)

for block in sampled_blocks:
    s_corrected_sec = np.asarray([
        float(row[s_time_key]) / 1000 + offset_ms / 1000
        for row in block["S"]
    ])
    v_sec = np.asarray([
        float(row["Time Since Start of Day (seconds)"])
        for row in block["V"]
    ])
    overlap = max(
        0.0,
        min(s_corrected_sec.max(), v_sec.max())
        - max(s_corrected_sec.min(), v_sec.min()),
    )
    print(
        f'{block["fraction"]:.0%}: '
        f'corrected S-M={s_corrected_sec.min():.1f}–'
        f'{s_corrected_sec.max():.1f}s; '
        f'V-M={v_sec.min():.1f}–{v_sec.max():.1f}s; '
        f'overlap={overlap:.1f}s'
    )

Candidate S-to-V offset: 29604.173 seconds
Original-prefix rows inside V time range: 4884
Match error p95/max: 1.0 100.0 ms
Within 100 ms: 4884 / 4884
25%: corrected S-M=32267.7–32830.0s; V-M=32228.1–32711.2s; overlap=443.5s
50%: corrected S-M=30493.9–31052.2s; V-M=34879.2–35369.5s; overlap=0.0s
75%: corrected S-M=33140.1–33699.2s; V-M=37553.3–38039.4s; overlap=0.0s


In [35]:
V_TIME_KEY = "Time Since Start of Day (seconds)"
S_TIME_KEY = "TIME SINCE START (ms)"
chunk_bytes = 1_572_864  # 1.5 MiB

# V-M byte-position / timestamp anchors from the prefix and sampled ranges.
anchors = [(0, float(v_times_ms_big[0]) / 1000)]
for block in sampled_blocks:
    byte_start = int(block["fraction"] * size_v)
    first_v_time = float(block["V"][0][V_TIME_KEY])
    anchors.append((byte_start, first_v_time))
anchors.sort(key=lambda item: item[1])

def estimate_v_byte(time_sec):
    for (b0, t0), (b1, t1) in zip(anchors[:-1], anchors[1:]):
        if t0 <= time_sec <= t1:
            return b0 + (time_sec - t0) * (b1 - b0) / (t1 - t0)
    raise ValueError(f"No V-M byte/time anchors cover {time_sec:.1f}s")

targeted_v_blocks = {}

for block in sampled_blocks:
    if block["fraction"] == 0.25:
        continue  # this V-M chunk already overlaps the S-M chunk

    s_times_sec = np.asarray([
        (float(row[S_TIME_KEY]) + offset_ms) / 1000
        for row in block["S"]
    ])
    desired_start = s_times_sec.min() - 150  # leave margin before the S-M chunk
    byte_start = max(0, int(estimate_v_byte(desired_start)))

    v_rows, content_range = read_csv_byte_range(
        V_URL, byte_start, chunk_bytes, v_fields
    )
    v_times_sec = np.asarray([
        float(row[V_TIME_KEY]) for row in v_rows
    ])

    overlap = max(
        0.0,
        min(s_times_sec.max(), v_times_sec.max())
        - max(s_times_sec.min(), v_times_sec.min())
    )
    errors_ms = nearest_ms_errors(
        s_times_sec * 1000,
        v_times_sec * 1000,
    )

    targeted_v_blocks[block["fraction"]] = v_rows

    print(
        f'{block["fraction"]:.0%}: V-M {content_range}; '
        f'S-M corrected={s_times_sec.min():.1f}–'
        f'{s_times_sec.max():.1f}s; '
        f'V-M={v_times_sec.min():.1f}–{v_times_sec.max():.1f}s; '
        f'overlap={overlap:.1f}s; '
        f'S rows within 100ms={np.sum(errors_ms <= 100)}/{len(errors_ms)}'
    )

50%: V-M bytes 1579772-3152635/22507823; S-M corrected=30493.9–31052.2s; V-M=30346.5–31076.0s; overlap=558.4s; S rows within 100ms=5572/5572
75%: V-M bytes 7244402-8817265/22507823; S-M corrected=33140.1–33699.2s; V-M=32986.3–33721.6s; overlap=559.1s; S rows within 100ms=5592/5592


In [36]:
W, H = 20, 10
S_TIME_KEY = "TIME SINCE START (ms)"
V_TIME_KEY = "Time Since Start of Day (seconds)"
SPEED_KEY = "Indicated Vehicle Speed (km/hr)"

def make_phone_speed_examples(s_rows, v_rows):
    v_ms = np.asarray(
        [float(row[V_TIME_KEY]) * 1000 for row in v_rows],
        dtype=float,
    )
    order = np.argsort(v_ms)
    v_ms = v_ms[order]
    v_sorted = [v_rows[i] for i in order]

    paired = []
    match_errors = []

    for s_row in s_rows:
        s_ms = float(s_row[S_TIME_KEY]) + offset_ms
        pos = int(np.searchsorted(v_ms, s_ms))
        candidates = [j for j in (pos - 1, pos) if 0 <= j < len(v_ms)]
        j = min(candidates, key=lambda k: abs(v_ms[k] - s_ms))
        error_ms = abs(v_ms[j] - s_ms)

        if error_ms > 100:
            continue

        phone = [float(s_row[col]) for col in phone_cols_big]
        ecu_speed_mps = float(v_sorted[j][SPEED_KEY]) / 3.6
        paired.append((s_ms, phone, ecu_speed_mps))
        match_errors.append(error_ms)

    paired.sort(key=lambda item: item[0])

    if not paired:
        return np.empty((0, W, 6)), np.empty(0), 0, np.empty(0)

    times = np.asarray([item[0] for item in paired])
    phone = np.asarray([item[1] for item in paired], dtype=np.float32)
    speeds = np.asarray([item[2] for item in paired], dtype=np.float32)

    # Split on gaps/duplicates so no window bridges missing samples.
    dt = np.diff(times)
    cuts = np.flatnonzero((dt > 150) | (dt < 50)) + 1
    runs = np.split(np.arange(len(times)), cuts)

    windows, targets = [], []
    for run in runs:
        if len(run) < W + H:
            continue
        for start in range(len(run) - W - H + 1):
            last_input = start + W - 1
            future = last_input + H
            windows.append(phone[run[start:start + W]])
            targets.append(speeds[run[future]])

    return (
        np.asarray(windows, dtype=np.float32),
        np.asarray(targets, dtype=np.float32),
        len(paired),
        np.asarray(match_errors),
    )

new_X, new_y = {}, {}

for block in sampled_blocks:
    fraction = round(float(block["fraction"]), 2)

    if fraction == 0.25:
        v_rows = block["V"]                 # partial overlap is filtered by timestamps
    else:
        v_rows = targeted_v_blocks[fraction]

    new_X[fraction], new_y[fraction], matched_n, errors = (
        make_phone_speed_examples(block["S"], v_rows)
    )

    print(
        f"{fraction:.0%}: matched rows={matched_n}, "
        f"windows={len(new_X[fraction])}, "
        f"match-error p95/max="
        f"{np.percentile(errors, 95):.1f}/"
        f"{errors.max():.1f} ms"
    )

# Keep the earlier prefix's original validation/tail out of this retraining.
X_train_expanded = np.concatenate(
    [X_speed_check[train_idx], new_X[0.50]], axis=0
)
y_train_expanded = np.concatenate(
    [speed_future_ecu[train_idx], new_y[0.50]]
)

X_val_expanded, y_val_expanded = new_X[0.25], new_y[0.25]
X_test_expanded, y_test_expanded = new_X[0.75], new_y[0.75]

print(
    "Expanded train / validation / test:",
    len(y_train_expanded), len(y_val_expanded), len(y_test_expanded)
)

# Normalize phone channels from the expanded training set only.
x_mu = X_train_expanded.mean(axis=(0, 1), keepdims=True)
x_sd = X_train_expanded.std(axis=(0, 1), keepdims=True)
x_sd[x_sd < 1e-6] = 1.0

def flatten_phone(x):
    return ((x - x_mu) / x_sd).reshape(len(x), -1).astype(np.float64)

A_train = np.column_stack([
    np.ones(len(X_train_expanded)),
    flatten_phone(X_train_expanded),
])
A_val = np.column_stack([
    np.ones(len(X_val_expanded)),
    flatten_phone(X_val_expanded),
])
A_test = np.column_stack([
    np.ones(len(X_test_expanded)),
    flatten_phone(X_test_expanded),
])

penalty = np.eye(A_train.shape[1])
penalty[0, 0] = 0.0

scores = []
for alpha in [0.1, 1.0, 10.0, 100.0]:
    coef = np.linalg.solve(
        A_train.T @ A_train + alpha * penalty,
        A_train.T @ y_train_expanded,
    )
    val_pred = A_val @ coef
    val_mae = np.mean(np.abs(val_pred - y_val_expanded))
    scores.append((val_mae, alpha, coef))
    print(f"alpha={alpha:5.1f} | validation MAE={val_mae * 3.6:.2f} km/h")

_, best_alpha_expanded, best_coef_expanded = min(
    scores, key=lambda item: item[0]
)

def report_expanded_speed(name, design, truth):
    pred = design @ best_coef_expanded
    baseline = np.full(len(truth), y_train_expanded.mean())
    print(
        f"{name}: Ridge MAE="
        f"{np.mean(np.abs(pred - truth)) * 3.6:.2f} km/h; "
        f"train-mean baseline="
        f"{np.mean(np.abs(baseline - truth)) * 3.6:.2f} km/h"
    )

print("Chosen alpha:", best_alpha_expanded)
report_expanded_speed("Validation", A_val, y_val_expanded)
report_expanded_speed("Test", A_test, y_test_expanded)

25%: matched rows=4436, windows=4407, match-error p95/max=1.0/9.0 ms
50%: matched rows=5572, windows=5514, match-error p95/max=49.0/50.0 ms
75%: matched rows=5592, windows=5563, match-error p95/max=49.0/50.0 ms
Expanded train / validation / test: 8883 4407 5563
alpha=  0.1 | validation MAE=19.86 km/h
alpha=  1.0 | validation MAE=19.86 km/h
alpha= 10.0 | validation MAE=19.86 km/h
alpha=100.0 | validation MAE=19.86 km/h
Chosen alpha: 100.0
Validation: Ridge MAE=19.86 km/h; train-mean baseline=20.58 km/h
Test: Ridge MAE=12.51 km/h; train-mean baseline=12.73 km/h


In [39]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

# Normalize using expanded training data only.
x_mu_exp = X_train_expanded.mean(axis=(0, 1), keepdims=True)
x_sd_exp = X_train_expanded.std(axis=(0, 1), keepdims=True)
x_sd_exp[x_sd_exp < 1e-6] = 1.0

def normalize_x_exp(x):
    return ((x - x_mu_exp) / x_sd_exp).astype(np.float32)

Xtr_exp_t = torch.from_numpy(normalize_x_exp(X_train_expanded))
Xva_exp_t = torch.from_numpy(normalize_x_exp(X_val_expanded))
Xte_exp_t = torch.from_numpy(normalize_x_exp(X_test_expanded))

y_mu_exp = y_train_expanded.mean()
y_sd_exp = y_train_expanded.std()
if y_sd_exp < 1e-6:
    y_sd_exp = 1.0

def normalize_y_exp(y):
    return ((y - y_mu_exp) / y_sd_exp).astype(np.float32)

ytr_exp_t = torch.from_numpy(normalize_y_exp(y_train_expanded)).unsqueeze(1)

loader_exp = DataLoader(
    TensorDataset(Xtr_exp_t, ytr_exp_t),
    batch_size=128,
    shuffle=True,
)

class ExpandedSpeedGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(6, 32, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(32, 1))

    def forward(self, x):
        sequence, _ = self.gru(x)
        return self.head(sequence[:, -1, :])

model_exp = ExpandedSpeedGRU()
optimizer_exp = torch.optim.AdamW(
    model_exp.parameters(), lr=5e-4, weight_decay=1e-4
)

best_mae_exp = float("inf")
best_state_exp = None
best_epoch_exp = 0
stale_exp = 0

for epoch in range(1, 41):
    model_exp.train()
    loss_sum = 0.0
    count = 0

    for xb, yb in loader_exp:
        optimizer_exp.zero_grad()
        pred = model_exp(xb)
        loss = F.smooth_l1_loss(pred, yb)
        loss.backward()
        optimizer_exp.step()
        loss_sum += loss.item() * len(xb)
        count += len(xb)

    model_exp.eval()
    with torch.no_grad():
        pva_norm = model_exp(Xva_exp_t).squeeze(1).numpy()

    pva = pva_norm * y_sd_exp + y_mu_exp
    val_mae_exp = np.mean(np.abs(pva - y_val_expanded))

    if val_mae_exp < best_mae_exp - 1e-4:
        best_mae_exp = val_mae_exp
        best_state_exp = {
            k: v.detach().clone()
            for k, v in model_exp.state_dict().items()
        }
        best_epoch_exp = epoch
        stale_exp = 0
    else:
        stale_exp += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:2d} | "
            f"train Huber={loss_sum / count:.4f} | "
            f"validation MAE={val_mae_exp * 3.6:.2f} km/h"
        )

    if stale_exp >= 8:
        break

model_exp.load_state_dict(best_state_exp)
print("Best epoch:", best_epoch_exp)

def report_expanded_gru(name, X_t, y):
    model_exp.eval()
    with torch.no_grad():
        pred_norm = model_exp(X_t).squeeze(1).numpy()

    pred = pred_norm * y_sd_exp + y_mu_exp
    baseline = np.full(len(y), y_train_expanded.mean())

    print(
        f"{name}: GRU MAE={np.mean(np.abs(pred - y)) * 3.6:.2f} km/h; "
        f"train-mean baseline="
        f"{np.mean(np.abs(baseline - y)) * 3.6:.2f} km/h"
    )

report_expanded_gru("Validation", Xva_exp_t, y_val_expanded)
report_expanded_gru("Test", Xte_exp_t, y_test_expanded)


Epoch  1 | train Huber=0.3708 | validation MAE=20.83 km/h
Epoch  5 | train Huber=0.2910 | validation MAE=14.37 km/h
Epoch 10 | train Huber=0.2281 | validation MAE=15.67 km/h
Best epoch: 5
Validation: GRU MAE=14.37 km/h; train-mean baseline=20.58 km/h
Test: GRU MAE=13.87 km/h; train-mean baseline=12.73 km/h


In [40]:
ACCEL_KEY = "Indicated Longitudinal Acceleration (g)"
S_TIME_KEY = "TIME SINCE START (ms)"
V_TIME_KEY = "Time Since Start of Day (seconds)"
W = 20

def make_accel_examples(s_rows, v_rows):
    v_times = np.asarray(
        [float(row[V_TIME_KEY]) * 1000 for row in v_rows],
        dtype=float,
    )
    v_accel = np.asarray(
        [float(row[ACCEL_KEY]) for row in v_rows],
        dtype=float,
    )

    valid = np.isfinite(v_times) & np.isfinite(v_accel)
    order = np.argsort(v_times[valid])
    v_times = v_times[valid][order]
    v_accel = v_accel[valid][order]

    # Remove duplicate V-M timestamps before interpolation.
    v_times, unique_idx = np.unique(v_times, return_index=True)
    v_accel = v_accel[unique_idx]

    paired = []
    for s_row in s_rows:
        s_time = float(s_row[S_TIME_KEY]) + offset_ms
        if s_time < v_times[0] or s_time > v_times[-1]:
            continue

        phone = [float(s_row[col]) for col in phone_cols_big]
        accel_target = np.interp(s_time, v_times, v_accel)
        paired.append((s_time, phone, accel_target))

    paired.sort(key=lambda item: item[0])
    times = np.asarray([item[0] for item in paired])
    phone = np.asarray([item[1] for item in paired], dtype=np.float32)
    targets = np.asarray([item[2] for item in paired], dtype=np.float32)

    cuts = np.flatnonzero((np.diff(times) > 150) | (np.diff(times) < 50)) + 1
    runs = np.split(np.arange(len(times)), cuts)

    windows, labels = [], []
    for run in runs:
        for start in range(len(run) - W + 1):
            end = start + W - 1
            windows.append(phone[run[start:start + W]])
            labels.append(targets[run[end]])

    return (
        np.asarray(windows, dtype=np.float32),
        np.asarray(labels, dtype=np.float32),
        len(paired),
    )

X_acc, y_acc = {}, {}

for block in sampled_blocks:
    fraction = round(float(block["fraction"]), 2)
    v_rows = (
        block["V"]
        if fraction == 0.25
        else targeted_v_blocks[fraction]
    )
    X_acc[fraction], y_acc[fraction], paired_n = make_accel_examples(
        block["S"], v_rows
    )
    print(
        f"{fraction:.0%}: paired rows={paired_n}, "
        f"windows={len(X_acc[fraction])}"
    )

# Chronological chunks: 50% train, 25% validation, 75% test.
Xtr_acc, ytr_acc = X_acc[0.50], y_acc[0.50]
Xva_acc, yva_acc = X_acc[0.25], y_acc[0.25]
Xte_acc, yte_acc = X_acc[0.75], y_acc[0.75]

# Normalize phone channels using training data only.
mu_acc = Xtr_acc.mean(axis=(0, 1), keepdims=True)
sd_acc = Xtr_acc.std(axis=(0, 1), keepdims=True)
sd_acc[sd_acc < 1e-6] = 1.0

def flat_acc(x):
    return ((x - mu_acc) / sd_acc).reshape(len(x), -1).astype(np.float64)

A_tr = np.column_stack([np.ones(len(Xtr_acc)), flat_acc(Xtr_acc)])
A_va = np.column_stack([np.ones(len(Xva_acc)), flat_acc(Xva_acc)])
A_te = np.column_stack([np.ones(len(Xte_acc)), flat_acc(Xte_acc)])

penalty = np.eye(A_tr.shape[1])
penalty[0, 0] = 0.0

scores = []
for alpha in [0.1, 1.0, 10.0, 100.0]:
    coef = np.linalg.solve(
        A_tr.T @ A_tr + alpha * penalty,
        A_tr.T @ ytr_acc,
    )
    val_pred = A_va @ coef
    val_mae = np.mean(np.abs(val_pred - yva_acc))
    scores.append((val_mae, alpha, coef))
    print(f"alpha={alpha:5.1f} | validation MAE={val_mae:.4f} g")

_, best_alpha_acc, best_coef_acc = min(scores, key=lambda item: item[0])

def report_acc(name, design, truth):
    pred = design @ best_coef_acc
    baseline = np.full(len(truth), ytr_acc.mean())
    mae = np.mean(np.abs(pred - truth))
    base_mae = np.mean(np.abs(baseline - truth))
    print(
        f"{name}: Ridge MAE={mae:.4f} g "
        f"({mae * 9.80665:.3f} m/s²); "
        f"train-mean baseline={base_mae:.4f} g"
    )

print("Chosen alpha:", best_alpha_acc)
report_acc("Validation", A_va, yva_acc)
report_acc("Test", A_te, yte_acc)

25%: paired rows=4435, windows=4416
50%: paired rows=5572, windows=5534
75%: paired rows=5592, windows=5573
alpha=  0.1 | validation MAE=0.0975 g
alpha=  1.0 | validation MAE=0.0974 g
alpha= 10.0 | validation MAE=0.0974 g
alpha=100.0 | validation MAE=0.0973 g
Chosen alpha: 100.0
Validation: Ridge MAE=0.0973 g (0.954 m/s²); train-mean baseline=0.0959 g
Test: Ridge MAE=0.0626 g (0.614 m/s²); train-mean baseline=0.0620 g


In [41]:
speed_splits_kmh = {
    "Train": np.concatenate([
        speed_future_ecu[train_idx],
        new_y[0.50],
    ]) * 3.6,
    "Validation": new_y[0.25] * 3.6,
    "Test": new_y[0.75] * 3.6,
}

for name, speeds in speed_splits_kmh.items():
    stationary = speeds <= 1.0
    moving = speeds >= 5.0
    ambiguous = ~(stationary | moving)

    print(
        f"{name}: stationary={stationary.sum()}, "
        f"moving={moving.sum()}, "
        f"ignored creep={ambiguous.sum()}"
    )

Train: stationary=341, moving=8258, ignored creep=284
Validation: stationary=204, moving=4108, ignored creep=95
Test: stationary=627, moving=4684, ignored creep=252


In [42]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# Phone-only inputs; ECU speed is used only to make labels.
X_train_state_raw = np.concatenate(
    [X_speed_check[train_idx], new_X[0.50]], axis=0
)
X_val_state_raw = new_X[0.25]
X_test_state_raw = new_X[0.75]

def make_state_labels(speed_kmh):
    # 1 = stationary, 0 = moving, -1 = ignored creep range.
    labels = np.full(len(speed_kmh), -1, dtype=np.int64)
    labels[speed_kmh <= 1.0] = 1
    labels[speed_kmh >= 5.0] = 0
    keep = labels >= 0
    return labels[keep], keep

y_train_state_all, keep_train = make_state_labels(speed_splits_kmh["Train"])
y_val_state_all, keep_val = make_state_labels(speed_splits_kmh["Validation"])
y_test_state_all, keep_test = make_state_labels(speed_splits_kmh["Test"])

X_train_state_raw = X_train_state_raw[keep_train]
X_val_state_raw = X_val_state_raw[keep_val]
X_test_state_raw = X_test_state_raw[keep_test]

print(
    "Used examples | train:",
    np.bincount(y_train_state_all, minlength=2),
    "| validation:",
    np.bincount(y_val_state_all, minlength=2),
    "| test:",
    np.bincount(y_test_state_all, minlength=2),
)
print("Class order above: [moving, stationary]")

# Normalize phone channels using training data only.
x_mu_state = X_train_state_raw.mean(axis=(0, 1), keepdims=True)
x_sd_state = X_train_state_raw.std(axis=(0, 1), keepdims=True)
x_sd_state[x_sd_state < 1e-6] = 1.0

def state_tensor(x):
    return torch.from_numpy(
        ((x - x_mu_state) / x_sd_state).astype(np.float32)
    )

Xtr_state_t = state_tensor(X_train_state_raw)
Xva_state_t = state_tensor(X_val_state_raw)
Xte_state_t = state_tensor(X_test_state_raw)

ytr_state_t = torch.from_numpy(y_train_state_all.astype(np.float32)).unsqueeze(1)
yva_state_t = torch.from_numpy(y_val_state_all.astype(np.float32)).unsqueeze(1)

n_moving = int(np.sum(y_train_state_all == 0))
n_stationary = int(np.sum(y_train_state_all == 1))
pos_weight_state = torch.tensor([n_moving / n_stationary])

train_loader_state = DataLoader(
    TensorDataset(Xtr_state_t, ytr_state_t),
    batch_size=128,
    shuffle=True,
)

class MotionStateGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(6, 32, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(32, 1))

    def forward(self, x):
        sequence, _ = self.gru(x)
        return self.head(sequence[:, -1, :])

torch.manual_seed(42)
state_model = MotionStateGRU()
optimizer_state = torch.optim.AdamW(
    state_model.parameters(), lr=5e-4, weight_decay=1e-4
)
loss_fn_state = nn.BCEWithLogitsLoss(pos_weight=pos_weight_state)

best_val_loss_state = float("inf")
best_state_weights = None
best_state_epoch = 0
stale_state = 0

for epoch in range(1, 41):
    state_model.train()
    loss_sum = 0.0
    count = 0

    for xb, yb in train_loader_state:
        optimizer_state.zero_grad()
        logits = state_model(xb)
        loss = loss_fn_state(logits, yb)
        loss.backward()
        optimizer_state.step()
        loss_sum += loss.item() * len(xb)
        count += len(xb)

    state_model.eval()
    with torch.no_grad():
        val_logits = state_model(Xva_state_t)
        val_loss = loss_fn_state(val_logits, yva_state_t).item()

    if val_loss < best_val_loss_state - 1e-4:
        best_val_loss_state = val_loss
        best_state_weights = {
            k: v.detach().clone()
            for k, v in state_model.state_dict().items()
        }
        best_state_epoch = epoch
        stale_state = 0
    else:
        stale_state += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:2d} | train loss={loss_sum / count:.4f} | "
            f"validation loss={val_loss:.4f}"
        )

    if stale_state >= 7:
        break

state_model.load_state_dict(best_state_weights)
print("Best epoch:", best_state_epoch)

def state_probabilities(x_tensor):
    state_model.eval()
    with torch.no_grad():
        return torch.sigmoid(state_model(x_tensor)).squeeze(1).numpy()

val_prob_state = state_probabilities(Xva_state_t)
test_prob_state = state_probabilities(Xte_state_t)

def state_metrics(y_true, y_pred):
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    balanced_accuracy = (recall + specificity) / 2

    return {
        "balanced_accuracy": balanced_accuracy,
        "stationary_precision": precision,
        "stationary_recall": recall,
        "stationary_F1": f1,
        "confusion_matrix_[TN,FP;FN,TP]": [[tn, fp], [fn, tp]],
    }

thresholds = np.linspace(0.05, 0.95, 37)
threshold = max(
    thresholds,
    key=lambda t: state_metrics(
        y_val_state_all, (val_prob_state >= t).astype(int)
    )["balanced_accuracy"],
)

print("Validation-selected threshold:", round(float(threshold), 2))
print("Validation:", state_metrics(
    y_val_state_all, (val_prob_state >= threshold).astype(int)
))
print("Held-out test:", state_metrics(
    y_test_state_all, (test_prob_state >= threshold).astype(int)
))
print("Test majority baseline: predict moving for every example")
print(state_metrics(y_test_state_all, np.zeros_like(y_test_state_all)))

Used examples | train: [8258  341] | validation: [4108  204] | test: [4684  627]
Class order above: [moving, stationary]
Epoch  1 | train loss=1.3230 | validation loss=1.4367
Epoch  5 | train loss=1.1700 | validation loss=1.3622
Epoch 10 | train loss=0.9409 | validation loss=1.6388
Best epoch: 6
Validation-selected threshold: 0.55
Validation: {'balanced_accuracy': 0.7359050728373141, 'stationary_precision': 0.10342717258261934, 'stationary_recall': 0.8284313725490197, 'stationary_F1': 0.1838955386289445, 'confusion_matrix_[TN,FP;FN,TP]': [[2643, 1465], [35, 169]]}
Held-out test: {'balanced_accuracy': 0.4915876368975385, 'stationary_precision': 0.11567068479097205, 'stationary_recall': 0.7192982456140351, 'stationary_F1': 0.1992929739284136, 'confusion_matrix_[TN,FP;FN,TP]': [[1236, 3448], [176, 451]]}
Test majority baseline: predict moving for every example
{'balanced_accuracy': 0.5, 'stationary_precision': 0.0, 'stationary_recall': 0.0, 'stationary_F1': 0.0, 'confusion_matrix_[TN,FP;F

In [43]:
threshold_grid = np.linspace(0.01, 0.99, 99)

f1_scores = [
    state_metrics(
        y_val_state_all,
        (val_prob_state >= t).astype(int),
    )["stationary_F1"]
    for t in threshold_grid
]

threshold_f1 = float(threshold_grid[np.argmax(f1_scores)])
print("F1-selected validation threshold:", round(threshold_f1, 2))
print("Validation:", state_metrics(
    y_val_state_all, (val_prob_state >= threshold_f1).astype(int)
))
print("Test:", state_metrics(
    y_test_state_all, (test_prob_state >= threshold_f1).astype(int)
))

F1-selected validation threshold: 0.56
Validation: {'balanced_accuracy': 0.7290270538595185, 'stationary_precision': 0.1083676268861454, 'stationary_recall': 0.7745098039215687, 'stationary_F1': 0.1901323706377858, 'confusion_matrix_[TN,FP;FN,TP]': [[2808, 1300], [46, 158]]}
Test: {'balanced_accuracy': 0.4958013094221463, 'stationary_precision': 0.11675977653631285, 'stationary_recall': 0.6666666666666666, 'stationary_F1': 0.1987164250059425, 'confusion_matrix_[TN,FP;FN,TP]': [[1522, 3162], [209, 418]]}


In [44]:
S_TIME = "TIME SINCE START (ms)"
V_TIME = "Time Since Start of Day (seconds)"
ECU_SPEED = "Indicated Vehicle Speed (km/hr)"
ACC_COLS = [
    col for col in phone_cols_big
    if "ACCELEROMETER" in col.upper()
]

def moving_average(x, width):
    width = max(1, int(width))
    if width % 2 == 0:
        width += 1
    return np.convolve(x, np.ones(width) / width, mode="same")

def corr_value(x, y):
    if len(x) < 30 or np.std(x) < 1e-8 or np.std(y) < 1e-8:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

def check_sync_physics(name, s_rows, v_rows):
    v_t = np.asarray([float(r[V_TIME]) for r in v_rows])
    v_speed = np.asarray([
        float(r[ECU_SPEED]) / 3.6 for r in v_rows
    ])

    order = np.argsort(v_t)
    v_t, v_speed = v_t[order], v_speed[order]
    v_t, unique_idx = np.unique(v_t, return_index=True)
    v_speed = v_speed[unique_idx]

    s_t = np.asarray([
        (float(r[S_TIME]) + offset_ms) / 1000
        for r in s_rows
    ])
    s_acc = np.asarray([
        [float(r[c]) for c in ACC_COLS]
        for r in s_rows
    ])

    order = np.argsort(s_t)
    s_t, s_acc = s_t[order], s_acc[order]

    keep = (s_t >= v_t[0]) & (s_t <= v_t[-1])
    s_t, s_acc = s_t[keep], s_acc[keep]

    if len(s_t) < 200:
        print(name, ": too little overlap")
        return

    dt = float(np.median(np.diff(s_t)))
    t = np.arange(s_t[0], s_t[-1], dt)

    acc = np.column_stack([
        np.interp(t, s_t, s_acc[:, j])
        for j in range(3)
    ])
    speed = np.interp(t, v_t, v_speed)

    # Approximate gravity with a centered ~2-second low-pass.
    gravity_width = max(3, int(round(2.0 / dt)))
    if gravity_width % 2 == 0:
        gravity_width += 1
    gravity = np.column_stack([
        moving_average(acc[:, j], gravity_width)
        for j in range(3)
    ])
    g_norm = np.linalg.norm(gravity, axis=1, keepdims=True)
    g_unit = gravity / np.maximum(g_norm, 1e-6)

    dynamic_acc = acc - gravity
    horizontal_acc = dynamic_acc - (
        np.sum(dynamic_acc * g_unit, axis=1, keepdims=True) * g_unit
    )

    # Smooth ECU speed before differentiating.
    speed_width = max(3, int(round(0.5 / dt)))
    speed_smooth = moving_average(speed, speed_width)
    ecu_accel = np.gradient(speed_smooth, dt)

    lag_max = int(round(2.0 / dt))
    edge = gravity_width // 2 + speed_width // 2
    results = []

    for axis in range(3):
        axis_results = []
        for lag in range(-lag_max, lag_max + 1):
            if lag >= 0:
                x = horizontal_acc[edge:len(t) - edge - lag, axis]
                y = ecu_accel[edge + lag:len(t) - edge]
            else:
                k = -lag
                x = horizontal_acc[edge + k:len(t) - edge, axis]
                y = ecu_accel[edge:len(t) - edge - k]

            r_all = corr_value(x, y)
            active = np.abs(y) > 0.2  # diagnostic only: dynamic samples
            r_active = corr_value(x[active], y[active])
            axis_results.append((lag, r_all, r_active, int(active.sum())))

        zero = next(item for item in axis_results if item[0] == 0)
        finite_all = [item for item in axis_results if np.isfinite(item[1])]
        finite_active = [item for item in axis_results if np.isfinite(item[2])]
        best_all = max(finite_all, key=lambda item: abs(item[1]))
        best_active = (
            max(finite_active, key=lambda item: abs(item[2]))
            if finite_active else None
        )

        print(
            f"  {ACC_COLS[axis]}: "
            f"zero-lag r={zero[1]:.3f}; "
            f"best all-sample |r|={abs(best_all[1]):.3f} "
            f"at {best_all[0] * dt:+.2f}s"
        )
        if best_active:
            print(
                f"    dynamic-only |r|={abs(best_active[2]):.3f} "
                f"at {best_active[0] * dt:+.2f}s "
                f"(n={best_active[3]})"
            )

    v_overlap = v_t[(v_t >= s_t[0]) & (v_t <= s_t[-1])]
    v_dt = np.diff(v_overlap)
    periods = np.asarray([
        float(r["Sample period (seconds)"])
        for r in v_rows
        if r.get("Sample period (seconds)", "") not in ("", None)
    ])

    print(
        f"{name}: n={len(t)}, phone dt median/p95="
        f"{np.median(np.diff(t)):.3f}/"
        f"{np.percentile(np.diff(t), 95):.3f}s; "
        f"V-M dt median/p95="
        f"{np.median(v_dt):.3f}/"
        f"{np.percentile(v_dt, 95):.3f}s; "
        f"V-M Sample period median="
        f"{np.median(periods):.3f}s; "
        f"estimated gravity magnitude median="
        f"{np.median(g_norm):.2f} m/s²"
    )

segments_to_check = [("prefix", s_big, v_big)]
for block in sampled_blocks:
    fraction = round(float(block["fraction"]), 2)
    v_rows = (
        block["V"] if fraction == 0.25
        else targeted_v_blocks[fraction]
    )
    segments_to_check.append(
        (f"{fraction:.0%} range", block["S"], v_rows)
    )

for name, s_rows, v_rows in segments_to_check:
    check_sync_physics(name, s_rows, v_rows)

  ACCELEROMETER X (m/s�): zero-lag r=0.003; best all-sample |r|=0.047 at +1.70s
    dynamic-only |r|=0.063 at +1.70s (n=1622)
  ACCELEROMETER Y (m/s�): zero-lag r=0.001; best all-sample |r|=0.053 at +0.80s
    dynamic-only |r|=0.086 at +0.80s (n=1631)
  ACCELEROMETER Z (m/s�): zero-lag r=-0.003; best all-sample |r|=0.017 at +1.20s
    dynamic-only |r|=0.061 at -1.80s (n=1634)
prefix: n=4884, phone dt median/p95=0.100/0.100s; V-M dt median/p95=0.100/0.100s; V-M Sample period median=0.100s; estimated gravity magnitude median=10.02 m/s²
  ACCELEROMETER X (m/s�): zero-lag r=0.005; best all-sample |r|=0.102 at +0.90s
    dynamic-only |r|=0.119 at +0.90s (n=2972)
  ACCELEROMETER Y (m/s�): zero-lag r=0.007; best all-sample |r|=0.125 at +0.80s
    dynamic-only |r|=0.141 at +0.80s (n=2973)
  ACCELEROMETER Z (m/s�): zero-lag r=0.108; best all-sample |r|=0.400 at +0.80s
    dynamic-only |r|=0.407 at +0.80s (n=2973)
25% range: n=4435, phone dt median/p95=0.100/0.100s; V-M dt median/p95=0.100/0.100

In [45]:
def audit_ecu_speed(name, v_rows):
    t = np.asarray([float(r[V_TIME]) for r in v_rows])
    speed = np.asarray([float(r[ECU_SPEED]) for r in v_rows])

    order = np.argsort(t)
    t, speed = t[order], speed[order]
    t, unique_idx = np.unique(t, return_index=True)
    speed = speed[unique_idx]

    dt = np.diff(t)
    period = np.asarray([
        float(r["Sample period (seconds)"])
        for r in v_rows
        if r.get("Sample period (seconds)", "") not in ("", None)
    ])

    # Round only for this diagnostic, to avoid counting tiny float noise
    # as a speed update.
    speed_q = np.round(speed, 2)
    delta = np.diff(speed_q)
    changed = np.abs(delta) >= 0.01
    moving_changed = changed & (np.maximum(speed[:-1], speed[1:]) >= 5.0)

    def update_stats(mask):
        update_times = t[1:][mask]
        gaps = np.diff(update_times)
        steps = np.abs(delta[mask])
        if len(gaps):
            gap_text = f"median interval between changes={np.median(gaps):.2f}s"
        else:
            gap_text = "too few changes to estimate update interval"
        if len(steps):
            vals, counts = np.unique(np.round(steps, 2), return_counts=True)
            top = sorted(zip(vals, counts), key=lambda x: -x[1])[:5]
            step_text = (
                f"abs step median/p90={np.median(steps):.2f}/"
                f"{np.percentile(steps, 90):.2f} km/h; "
                f"common step sizes={[(float(v), int(c)) for v, c in top]}"
            )
        else:
            step_text = "no speed changes detected"
        return gap_text, step_text

    all_gap, all_steps = update_stats(changed)
    moving_gap, moving_steps = update_stats(moving_changed)

    print(
        f"{name}: row dt median/p95={np.median(dt):.3f}/"
        f"{np.percentile(dt, 95):.3f}s; "
        f"reported period median={np.median(period):.3f}s; "
        f"speed changes on {changed.mean():.1%} of intervals"
    )
    print(f"  all speeds: {all_gap}; {all_steps}")
    print(f"  while ≥5 km/h: {moving_gap}; {moving_steps}")

for name, _, v_rows in segments_to_check:
    audit_ecu_speed(name, v_rows)

prefix: row dt median/p95=0.100/0.100s; reported period median=0.100s; speed changes on 91.9% of intervals
  all speeds: median interval between changes=0.10s; abs step median/p90=0.08/0.25 km/h; common step sizes=[(0.02, 408), (0.03, 372), (0.04, 346), (0.05, 321), (0.06, 258)]
  while ≥5 km/h: median interval between changes=0.10s; abs step median/p90=0.08/0.25 km/h; common step sizes=[(0.02, 408), (0.03, 372), (0.04, 346), (0.05, 321), (0.06, 257)]
25% range: row dt median/p95=0.100/0.100s; reported period median=0.100s; speed changes on 92.6% of intervals
  all speeds: median interval between changes=0.10s; abs step median/p90=0.21/0.73 km/h; common step sizes=[(0.05, 163), (0.02, 151), (0.03, 147), (0.04, 136), (0.1, 134)]
  while ≥5 km/h: median interval between changes=0.10s; abs step median/p90=0.21/0.72 km/h; common step sizes=[(0.05, 160), (0.02, 147), (0.03, 146), (0.04, 136), (0.1, 131)]
50% range: row dt median/p95=0.100/0.100s; reported period median=0.100s; speed changes

In [46]:
print("S-M columns:", list(s_big[0].keys()))
print("V-M columns:", list(v_big[0].keys()))

S-M columns: ['GPS LATITUDE (degrees)', 'GPS LONGITUDE (degrees)', 'GPS ALTITUDE (m)', 'GPS SPEED (Kmh)', 'GPS ACCURACY (m)', 'GPS ORIENTATION (°)', 'GPS SATELLITES IN RANGE', 'TIME SINCE START (ms)', 'DATE (YYYY-MO-DD HH-MI-SS_SSS)', 'ACCELEROMETER X (m/s�)', 'ACCELEROMETER Y (m/s�)', 'ACCELEROMETER Z (m/s�)', 'GRAVITY X (m/s�)', 'GRAVITY Y (m/s�)', 'GRAVITY Z (m/s�)', 'GYROSCOPE Yaw (rad/s)', 'GYROSCOPE Pitch (rad/s)', 'GYROSCOPE Roll (rad/s)', 'MAGNETIC FIELD X (μT)', 'MAGNETIC FIELD Y (μT)', 'MAGNETIC FIELD Z (μT)', 'ORIENTATION (Yaw) (°)', 'ORIENTATION (Pitch) (°)', 'ORIENTATION (Roll ) (°)']
V-M columns: ['No of GPS Satellites Available', 'Time Since Start of Day (seconds)', 'Latitude (degrees)', 'Longitude (degrees)', 'Velocity (km/hr)', 'Heading (degrees)', 'Height (km)', 'Vertical velocity (km/hr)', 'Sample period (seconds)', 'Steering Angle (degrees)', 'Wheel Speed Front Left (rad/sec)', 'Wheel Speed Front Right (rad/sec)', 'Wheel Speed Rear Left (rad/sec)', 'Wheel Speed Rear

In [47]:
PHONE_TIME = "TIME SINCE START (ms)"
PHONE_GYRO = [
    "GYROSCOPE Yaw (rad/s)",
    "GYROSCOPE Pitch (rad/s)",
    "GYROSCOPE Roll (rad/s)",
]

VM_TIME = "Time Since Start of Day (seconds)"
VM_HEADING = "Heading (degrees)"
VM_GPS_SPEED = "Velocity (km/hr)"
VM_SATELLITES = "No of GPS Satellites Available"

def as_float(row, col):
    try:
        return float(row[col])
    except (KeyError, TypeError, ValueError):
        return np.nan

def safe_corr(x, y):
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 30 or np.std(x) < 1e-8 or np.std(y) < 1e-8:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

def check_gyro_vs_gps_heading(name, s_rows, v_rows):
    # Put phone and vehicle samples on the already-validated common time base.
    phone_t = np.asarray([
        (as_float(r, PHONE_TIME) + offset_ms) / 1000
        for r in s_rows
    ])
    gyro = np.asarray([
        [as_float(r, c) for c in PHONE_GYRO]
        for r in s_rows
    ]) * (180 / np.pi)  # rad/s -> deg/s

    psort = np.argsort(phone_t)
    phone_t, gyro = phone_t[psort], gyro[psort]
    phone_ok = np.isfinite(phone_t) & np.all(np.isfinite(gyro), axis=1)
    phone_t, gyro = phone_t[phone_ok], gyro[phone_ok]

    v_t = np.asarray([as_float(r, VM_TIME) for r in v_rows])
    heading = np.asarray([as_float(r, VM_HEADING) for r in v_rows])
    gps_speed = np.asarray([as_float(r, VM_GPS_SPEED) for r in v_rows])
    satellites = np.asarray([as_float(r, VM_SATELLITES) for r in v_rows])

    vsort = np.argsort(v_t)
    v_t, heading = v_t[vsort], heading[vsort]
    gps_speed, satellites = gps_speed[vsort], satellites[vsort]

    v_t, unique_idx = np.unique(v_t, return_index=True)
    heading = heading[unique_idx]
    gps_speed = gps_speed[unique_idx]
    satellites = satellites[unique_idx]

    # Require GPS availability, speed > 5 m/s, and a plausible degree heading.
    gps_valid = (
        np.isfinite(v_t)
        & np.isfinite(heading) & (heading >= 0) & (heading <= 360)
        & np.isfinite(gps_speed) & (gps_speed > 18.0)
        & np.isfinite(satellites) & (satellites > 0)
    )
    heading = heading % 360

    # Circular mean of valid headings within each one-second bin.
    origin = np.floor(v_t[0])
    bin_id = np.floor(v_t - origin).astype(int)
    bin_t, bin_heading = [], []

    for b in np.unique(bin_id):
        m = (bin_id == b) & gps_valid
        if not np.any(m):
            continue
        rad = np.deg2rad(heading[m])
        mean_h = np.degrees(np.arctan2(
            np.mean(np.sin(rad)), np.mean(np.cos(rad))
        )) % 360
        bin_t.append(np.mean(v_t[m]))
        bin_heading.append(mean_h)

    bin_t = np.asarray(bin_t)
    bin_heading = np.asarray(bin_heading)

    gps_rates, gyro_rates = [], []
    for i in range(len(bin_t) - 1):
        t0, t1 = bin_t[i], bin_t[i + 1]
        dt = t1 - t0

        # Don't bridge likely GPS outages or large gaps.
        if not (0.5 <= dt <= 1.5):
            continue

        # Shortest signed heading change, correctly handling 0°/360° wrap.
        d_heading = (bin_heading[i + 1] - bin_heading[i] + 180) % 360 - 180
        gps_rate = d_heading / dt

        phone_interval = (phone_t > t0) & (phone_t <= t1)
        if phone_interval.sum() < 5:
            continue

        gps_rates.append(gps_rate)
        gyro_rates.append(np.mean(gyro[phone_interval], axis=0))

    gps_rates = np.asarray(gps_rates)
    gyro_rates = np.asarray(gyro_rates)

    print(
        f"{name}: GPS-valid samples={gps_valid.sum()}/{len(v_t)}; "
        f"valid 1s heading bins={len(bin_t)}; "
        f"usable rate intervals={len(gps_rates)}"
    )

    if len(gps_rates) == 0:
        return

    turns = np.abs(gps_rates) > 1.0
    print(
        f"  GPS heading-rate median |rate|="
        f"{np.median(np.abs(gps_rates)):.2f}°/s; "
        f"turn intervals |rate|>1°/s: {turns.sum()}"
    )

    for j, col in enumerate(PHONE_GYRO):
        r_all = safe_corr(gyro_rates[:, j], gps_rates)
        r_turns = safe_corr(gyro_rates[turns, j], gps_rates[turns])
        print(
            f"  {col}: r(all)={r_all:.3f}; "
            f"r(turn intervals)={r_turns:.3f}"
        )

for name, s_rows, v_rows in segments_to_check:
    check_gyro_vs_gps_heading(name, s_rows, v_rows)

prefix: GPS-valid samples=4643/4883; valid 1s heading bins=469; usable rate intervals=466
  GPS heading-rate median |rate|=6.24°/s; turn intervals |rate|>1°/s: 378
  GYROSCOPE Yaw (rad/s): r(all)=-0.677; r(turn intervals)=-0.714
  GYROSCOPE Pitch (rad/s): r(all)=-0.986; r(turn intervals)=-0.985
  GYROSCOPE Roll (rad/s): r(all)=0.763; r(turn intervals)=0.788
25% range: GPS-valid samples=4323/4832; valid 1s heading bins=437; usable rate intervals=393
  GPS heading-rate median |rate|=1.71°/s; turn intervals |rate|>1°/s: 250
  GYROSCOPE Yaw (rad/s): r(all)=-0.071; r(turn intervals)=-0.093
  GYROSCOPE Pitch (rad/s): r(all)=-0.739; r(turn intervals)=-0.759
  GYROSCOPE Roll (rad/s): r(all)=0.222; r(turn intervals)=0.260
50% range: GPS-valid samples=6089/7296; valid 1s heading bins=623; usable rate intervals=439
  GPS heading-rate median |rate|=1.13°/s; turn intervals |rate|>1°/s: 231
  GYROSCOPE Yaw (rad/s): r(all)=0.083; r(turn intervals)=0.119
  GYROSCOPE Pitch (rad/s): r(all)=0.056; r(turn

In [48]:
S_GPS_HEADING = "GPS ORIENTATION (°)"
S_GPS_SPEED = "GPS SPEED (Kmh)"
S_GPS_SATELLITES = "GPS SATELLITES IN RANGE"
S_GPS_ACCURACY = "GPS ACCURACY (m)"

def circular_mean_deg(degrees):
    radians = np.deg2rad(degrees)
    return np.degrees(np.arctan2(
        np.mean(np.sin(radians)), np.mean(np.cos(radians))
    )) % 360

def make_heading_bins(times, headings, speeds, satellites, origin):
    valid = (
        np.isfinite(times)
        & np.isfinite(headings) & (headings >= 0) & (headings <= 360)
        & np.isfinite(speeds) & (speeds > 18.0)  # >5 m/s
        & np.isfinite(satellites) & (satellites > 0)
    )
    headings = headings % 360
    bin_ids = np.floor(times - origin).astype(int)
    result = {}

    for b in np.unique(bin_ids[valid]):
        m = valid & (bin_ids == b)
        result[int(b)] = (
            float(np.mean(times[m])),
            float(circular_mean_deg(headings[m]))
        )
    return result, valid

def wrap_degrees(delta):
    return (delta + 180) % 360 - 180

def corr_if_enough(a, b):
    ok = np.isfinite(a) & np.isfinite(b)
    a, b = a[ok], b[ok]
    if len(a) < 30 or np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def compare_gps_headings(name, s_rows, v_rows):
    s_t = np.asarray([
        (as_float(r, PHONE_TIME) + offset_ms) / 1000
        for r in s_rows
    ])
    s_h = np.asarray([as_float(r, S_GPS_HEADING) for r in s_rows])
    s_speed = np.asarray([as_float(r, S_GPS_SPEED) for r in s_rows])
    s_sat = np.asarray([as_float(r, S_GPS_SATELLITES) for r in s_rows])
    s_acc = np.asarray([as_float(r, S_GPS_ACCURACY) for r in s_rows])

    v_t = np.asarray([as_float(r, VM_TIME) for r in v_rows])
    v_h = np.asarray([as_float(r, VM_HEADING) for r in v_rows])
    v_speed = np.asarray([as_float(r, VM_GPS_SPEED) for r in v_rows])
    v_sat = np.asarray([as_float(r, VM_SATELLITES) for r in v_rows])

    origin = np.floor(min(np.nanmin(s_t), np.nanmin(v_t)))
    s_bins, s_valid = make_heading_bins(s_t, s_h, s_speed, s_sat, origin)
    v_bins, v_valid = make_heading_bins(v_t, v_h, v_speed, v_sat, origin)

    common = sorted(set(s_bins) & set(v_bins))
    print(
        f"{name}: valid S-M GPS samples={s_valid.sum()}/{len(s_t)}; "
        f"valid V-M GPS samples={v_valid.sum()}/{len(v_t)}; "
        f"shared 1s bins={len(common)}"
    )

    valid_accuracy = s_acc[s_valid & np.isfinite(s_acc)]
    if len(valid_accuracy):
        print(
            f"  S-M GPS accuracy on valid samples: "
            f"median/p90={np.median(valid_accuracy):.1f}/"
            f"{np.percentile(valid_accuracy, 90):.1f} m"
        )

    if not common:
        return

    heading_diff = np.asarray([
        wrap_degrees(s_bins[b][1] - v_bins[b][1])
        for b in common
    ])
    bias = np.degrees(np.arctan2(
        np.mean(np.sin(np.deg2rad(heading_diff))),
        np.mean(np.cos(np.deg2rad(heading_diff)))
    ))
    print(
        f"  matched heading difference: circular bias={bias:.1f}°; "
        f"median absolute difference={np.median(np.abs(heading_diff)):.1f}°"
    )

    s_rates, v_rates = [], []
    for b0, b1 in zip(common[:-1], common[1:]):
        if b1 != b0 + 1:  # don't bridge missing one-second bins
            continue

        s_dt = s_bins[b1][0] - s_bins[b0][0]
        v_dt = v_bins[b1][0] - v_bins[b0][0]
        if not (0.5 <= s_dt <= 1.5 and 0.5 <= v_dt <= 1.5):
            continue

        s_rates.append(wrap_degrees(s_bins[b1][1] - s_bins[b0][1]) / s_dt)
        v_rates.append(wrap_degrees(v_bins[b1][1] - v_bins[b0][1]) / v_dt)

    s_rates, v_rates = np.asarray(s_rates), np.asarray(v_rates)
    if len(s_rates):
        print(
            f"  consecutive-bin heading rates: n={len(s_rates)}, "
            f"correlation={corr_if_enough(s_rates, v_rates):.3f}; "
            f"median absolute rate difference="
            f"{np.median(np.abs(s_rates - v_rates)):.2f}°/s"
        )

for name, s_rows, v_rows in segments_to_check:
    compare_gps_headings(name, s_rows, v_rows)

prefix: valid S-M GPS samples=0/5642; valid V-M GPS samples=4644/4884; shared 1s bins=0
25% range: valid S-M GPS samples=0/5624; valid V-M GPS samples=4323/4832; shared 1s bins=0
50% range: valid S-M GPS samples=0/5572; valid V-M GPS samples=6089/7296; shared 1s bins=0
75% range: valid S-M GPS samples=0/5592; valid V-M GPS samples=5487/7354; shared 1s bins=0


In [49]:
S_GPS_HEADING = "GPS ORIENTATION (°)"
S_GPS_SPEED = "GPS SPEED (Kmh)"
S_GPS_SATELLITES = "GPS SATELLITES IN RANGE"
S_GPS_ACCURACY = "GPS ACCURACY (m)"

def to_number(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan

def inspect_s_gps(name, rows):
    heading = np.asarray([to_number(r.get(S_GPS_HEADING)) for r in rows])
    speed = np.asarray([to_number(r.get(S_GPS_SPEED)) for r in rows])
    sats = np.asarray([to_number(r.get(S_GPS_SATELLITES)) for r in rows])
    accuracy = np.asarray([to_number(r.get(S_GPS_ACCURACY)) for r in rows])

    def describe(label, values):
        finite = values[np.isfinite(values)]
        if len(finite) == 0:
            print(f"  {label}: no numeric values")
            return
        print(
            f"  {label}: finite={len(finite)}/{len(values)}, "
            f"min/median/p90/max="
            f"{np.min(finite):.3f}/"
            f"{np.median(finite):.3f}/"
            f"{np.percentile(finite, 90):.3f}/"
            f"{np.max(finite):.3f}"
        )

    heading_ok = np.isfinite(heading) & (heading >= 0) & (heading <= 360)
    speed_ok = np.isfinite(speed) & (speed > 18)
    sats_ok = np.isfinite(sats) & (sats > 0)

    print(
        f"{name}: "
        f"heading 0–360={heading_ok.sum()}, "
        f"GPS speed >18 km/h={speed_ok.sum()}, "
        f"satellites >0={sats_ok.sum()}, "
        f"all three={np.sum(heading_ok & speed_ok & sats_ok)}"
    )
    describe("GPS orientation (degrees)", heading)
    describe("GPS speed (km/h)", speed)
    describe("satellites in range", sats)
    describe("GPS accuracy (m)", accuracy)

for name, s_rows, _ in segments_to_check:
    inspect_s_gps(name, s_rows)

print("Prefix raw values:")
for row in s_big[:5]:
    print({
        key: row.get(key)
        for key in [
            S_GPS_HEADING, S_GPS_SPEED,
            S_GPS_SATELLITES, S_GPS_ACCURACY
        ]
    })

prefix: heading 0–360=5642, GPS speed >18 km/h=270, satellites >0=0, all three=0
  GPS orientation (degrees): finite=5642/5642, min/median/p90/max=6.100/279.080/342.650/359.750
  GPS speed (km/h): finite=5642/5642, min/median/p90/max=0.000/8.140/17.310/18.620
  satellites in range: no numeric values
  GPS accuracy (m): finite=5642/5642, min/median/p90/max=2.000/3.000/4.000/8.000
25% range: heading 0–360=5624, GPS speed >18 km/h=653, satellites >0=0, all three=0
  GPS orientation (degrees): finite=5624/5624, min/median/p90/max=46.910/158.500/231.060/271.000
  GPS speed (km/h): finite=5624/5624, min/median/p90/max=0.000/10.690/18.860/22.130
  satellites in range: no numeric values
  GPS accuracy (m): finite=5624/5624, min/median/p90/max=2.000/3.000/4.000/8.000
50% range: heading 0–360=5572, GPS speed >18 km/h=1261, satellites >0=0, all three=0
  GPS orientation (degrees): finite=5572/5572, min/median/p90/max=2.940/299.840/326.650/353.000
  GPS speed (km/h): finite=5572/5572, min/median/p